## Observaciones Meta 8

### Con conjunto de datos Ing. Hada Natalia

In [ ]:
import re
import openpyxl

def analizar_observacion_estricta(texto_crudo):
    texto = str(texto_crudo).strip() if texto_crudo else ""
    
    # 1. DICCIONARIO DE ESTADOS Y ETAPAS (Basado en la directriz SIPSE)
    # La llave es el patrón Regex tolerante a ortografía; el valor es una tupla (Estado Oficial, Etapa)
    mapeo_sipse = {
        r"actualizaci[oó]n de criterios de elegibilidad y viabilidad": ("Actualización de criterios", "Proyecto de Inversión"),
        r"asistencia t[eé]cnica del sector": ("Asistencia técnica del sector", "Proyecto de Inversión"),
        r"cocreaci[oó]n de propuestas priorizadas.*": ("Cocreación de propuestas", "Proyecto de Inversión"),
        r"actualizaci[oó]n documento t[eé]cnico de soporte.*": ("Actualización DTS/Ficha EBI", "Proyecto de Inversión"),
        
        r"elaboraci[oó]n de estudios previos y documentos anexos": ("Estudios Previos y anexos", "Precontractual"),
        r"comit[eé] de contrataci[oó]n": ("Comité de Contratación", "Precontractual"),
        r"asistencia t[eé]cnica \(nivel central\)": ("Asistencia Técnica (Central)", "Precontractual"),
        r"publicaci[oó]n proceso contractual": ("Publicación proceso", "Precontractual"),
        r"adjudicaci[oó]n y suscripci[oó]n de proceso contractual": ("Adjudicación y suscripción", "Precontractual"),
        
        r"inicio ejecuci[oó]n del contrato": ("Inicio ejecución contrato", "Ejecución Contractual"),
        r"en ejecuci[oó]n \(\d{1,3}%\s*avance\)": ("En ejecución", "Ejecución Contractual"),
        r"en modificaci[oó]n contractual": ("En Modificación contractual", "Ejecución Contractual"),
        
        r"finalizaci[oó]n ejecuci[oó]n contrato": ("Finalización ejecución", "Postcontractual"),
        r"en proceso de liquidaci[oó]n": ("En proceso de liquidación", "Postcontractual")
    }
    
    cumple_estructura = "No"
    estado_extraido = "Sin Estado"
    etapa_extraida = "Sin Etapa"
    
    # 2. EVALUAR SI CUMPLE LA ESTRUCTURA Y EXTRAER
    for patron, (nombre_estado, nombre_etapa) in mapeo_sipse.items():
        # Busca si el texto inicia con el estado (permitiendo que opcionalmente pongan "1. ", "11-", etc.)
        if re.search(r"(?i)^(?:\d{1,2}\.?\s*-?\s*)?(" + patron + r")", texto):
            cumple_estructura = "Sí"
            estado_extraido = nombre_estado
            etapa_extraida = nombre_etapa
            break # Si encuentra el estado correcto, detiene la búsqueda
            
    # 3. EXTRAER EVIDENCIA CONDICIONAL
    patron_evidencia = r"(?i)(http[s]?://|drive\.google|anexo|radicado|pdf|acta|soporte)"
    if re.search(patron_evidencia, texto):
        tiene_evidencia = "Sí (Detectada)"
    else:
        tiene_evidencia = "No (Falta)"
            
    return cumple_estructura, etapa_extraida, estado_extraido, tiene_evidencia

def procesar_sabana_observaciones(ruta_excel):
    print("Leyendo Excel (Modo seguro para conservar logos y diseño)...")
    wb = openpyxl.load_workbook(ruta_excel)
    ws = wb.active
    
    # Lista de columnas que no pueden estar vacías
    nombres_requeridos = ["ENTIDAD", "VIGENCIA", "FECHA_CREACION", "DESCRIPCION_META", "META", "MES", "NUMPROYECTO", "PROYECTO"]
    indices_requeridos = {}
    fila_cabecera = None
    col_obs = None
    
    # 1. Buscar dinámicamente las columnas en el Excel
    for r in range(1, 16):
        for c in range(1, ws.max_column + 1):
            valor_celda = str(ws.cell(row=r, column=c).value).strip().upper()
            if valor_celda == "OBSERVACIONES":
                fila_cabecera = r
                col_obs = c
            elif valor_celda in nombres_requeridos:
                indices_requeridos[valor_celda] = c
        if fila_cabecera: 
            break
            
    if not fila_cabecera:
        print("Error: No se encontró la columna OBSERVACIONES.")
        return

    print(f"Cabeceras encontradas en fila {fila_cabecera}. Preparando estructura...")
    
    # 2. Insertar las 4 columnas nuevas
    ws.insert_cols(col_obs + 1, amount=4)
    ws.cell(row=fila_cabecera, column=col_obs + 1, value="Cumple_Estructura")
    ws.cell(row=fila_cabecera, column=col_obs + 2, value="Etapa_Extraida")
    ws.cell(row=fila_cabecera, column=col_obs + 3, value="Estado_Extraido")
    ws.cell(row=fila_cabecera, column=col_obs + 4, value="Evidencia_Validada")
    
    print("Analizando textos y validando columnas vacías...")
    
    # 3. Iterar y procesar desde la fila siguiente a los títulos
    for row in range(fila_cabecera + 1, ws.max_row + 1):
        
        # VALIDACIÓN CONDICIONAL: Revisar si alguna columna requerida está vacía
        fila_incompleta = False
        for nombre_col, col_idx in indices_requeridos.items():
            valor = ws.cell(row=row, column=col_idx).value
            if valor is None or str(valor).strip() == "" or str(valor).strip().lower() == "nan":
                fila_incompleta = True
                break
        
        # Si falta información clave en esa fila, llenamos con "-" y saltamos a la siguiente
        if fila_incompleta:
            ws.cell(row=row, column=col_obs + 1, value="-")
            ws.cell(row=row, column=col_obs + 2, value="-")
            ws.cell(row=row, column=col_obs + 3, value="-")
            ws.cell(row=row, column=col_obs + 4, value="-")
            continue
            
        # Si la fila tiene los datos completos, procesamos la observación
        texto_crudo = ws.cell(row=row, column=col_obs).value
        cumple, etapa, estado, evidencia = analizar_observacion_estricta(texto_crudo)
        
        ws.cell(row=row, column=col_obs + 1, value=cumple)
        ws.cell(row=row, column=col_obs + 2, value=etapa)
        ws.cell(row=row, column=col_obs + 3, value=estado)
        ws.cell(row=row, column=col_obs + 4, value=evidencia)
        
    # 4. Guardar copia limpia
    nueva_ruta = ruta_excel.replace(".xlsx", "_Analizado.xlsx")
    wb.save(nueva_ruta)
    print(f"¡Análisis completado! Archivo guardado en:\n{nueva_ruta}")

# Ejecutar
if __name__ == "__main__":
    # Cambia esto por tu ruta
    ruta = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\reporte_indicadores Julio.xlsx"
    procesar_sabana_observaciones(ruta)

Leyendo Excel (Modo seguro para conservar logos y diseño)...
Cabeceras encontradas en fila 1. Preparando estructura...
Analizando textos y validando columnas vacías...
¡Análisis completado! Archivo guardado en:
C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\reporte_indicadores Julio_Analizado.xlsx


### Versión 1.2

In [ ]:
import re
import unicodedata
import openpyxl
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
import pandas as pd

def analizar_observacion_estricta(texto_crudo):
    texto = str(texto_crudo).strip() if texto_crudo else ""
    
    # 1. DICCIONARIO DE ESTADOS Y ETAPAS
    mapeo_sipse = {
        r"actualizaci[oó]n de criterios de elegibilidad y viabilidad": ("Actualización de criterios", "Proyecto de Inversión"),
        r"asistencia t[eé]cnica del sector": ("Asistencia técnica del sector", "Proyecto de Inversión"),
        r"cocreaci[oó]n de propuestas priorizadas.*": ("Cocreación de propuestas", "Proyecto de Inversión"),
        r"actualizaci[oó]n documento t[eé]cnico de soporte.*": ("Actualización DTS/Ficha EBI", "Proyecto de Inversión"),
        
        r"elaboraci[oó]n de estudios previos y documentos anexos": ("Estudios Previos y anexos", "Precontractual"),
        r"comit[eé] de contrataci[oó]n": ("Comité de Contratación", "Precontractual"),
        r"asistencia t[eé]cnica \(nivel central\)": ("Asistencia Técnica (Central)", "Precontractual"),
        r"publicaci[oó]n proceso contractual": ("Publicación proceso", "Precontractual"),
        r"adjudicaci[oó]n y suscripci[oó]n de proceso contractual": ("Adjudicación y suscripción", "Precontractual"),
        
        r"inicio ejecuci[oó]n del contrato": ("Inicio ejecución contrato", "Ejecución Contractual"),
        r"en ejecuci[oó]n \(\d{1,3}%\s*avance\)": ("En ejecución", "Ejecución Contractual"),
        r"en modificaci[oó]n contractual": ("En Modificación contractual", "Ejecución Contractual"),
        
        r"finalizaci[oó]n ejecuci[oó]n contrato": ("Finalización ejecución", "Postcontractual"),
        r"en proceso de liquidaci[oó]n": ("En proceso de liquidación", "Postcontractual")
    }
    
    cumple_estructura = "No"
    estado_extraido = "Sin Estado"
    etapa_extraida = "Sin Etapa"
    
    # 2. EVALUAR ESTRUCTURA Y EXTRAER
    for patron, (nombre_estado, nombre_etapa) in mapeo_sipse.items():
        if re.search(r"(?i)^(?:\d{1,2}\.?\s*-?\s*)?(" + patron + r")", texto):
            cumple_estructura = "Sí"
            estado_extraido = nombre_estado
            etapa_extraida = nombre_etapa
            break 

        # Nota: Ya no extraemos la evidencia aquí, solo retornamos 3 variables
    return cumple_estructura, etapa_extraida, estado_extraido

def limpiar_nombre_localidad(entidad):
    """Estandariza el nombre de la localidad en mayúsculas exactas para evitar fallos de cruce."""
    if not isinstance(entidad, str):
        return ""
    
    entidad_str = entidad.strip().upper()
    nfkd = unicodedata.normalize('NFKD', entidad_str)
    entidad_limpia = "".join([c for c in nfkd if not unicodedata.combining(c)])
    
    if "USAQUEN" in entidad_limpia: return "01 - USAQUÉN"
    if "CHAPINERO" in entidad_limpia: return "02 - CHAPINERO"
    if "SANTA FE" in entidad_limpia: return "03 - SANTA FE"
    if "SAN CRISTOBAL" in entidad_limpia: return "04 - SAN CRISTÓBAL"
    if "USME" in entidad_limpia: return "05 - USME"
    if "TUNJUELITO" in entidad_limpia: return "06 - TUNJUELITO"
    if "BOSA" in entidad_limpia: return "07 - BOSA"
    if "KENNEDY" in entidad_limpia: return "08 - KENNEDY"
    if "FONTIBON" in entidad_limpia: return "09 - FONTIBÓN"
    if "ENGATIVA" in entidad_limpia: return "10 - ENGATIVÁ"
    if "SUBA" in entidad_limpia: return "11 - SUBA"
    if "BARRIOS UNIDOS" in entidad_limpia: return "12 - BARRIOS UNIDOS"
    if "TEUSAQUILLO" in entidad_limpia: return "13 - TEUSAQUILLO"
    if "MARTIRES" in entidad_limpia: return "14 - LOS MÁRTIRES"
    if "ANTONIO NARINO" in entidad_limpia: return "15 - ANTONIO NARIÑO"
    if "PUENTE ARANDA" in entidad_limpia: return "16 - PUENTE ARANDA"
    if "CANDELARIA" in entidad_limpia: return "17 - LA CANDELARIA"
    if "RAFAEL URIBE" in entidad_limpia: return "18 - RAFAEL URIBE URIBE"
    if "CIUDAD BOLIVAR" in entidad_limpia: return "19 - CIUDAD BOLÍVAR"
    if "SUMAPAZ" in entidad_limpia: return "20 - SUMAPAZ"
    
    return entidad_str

def evaluar_evidencia(texto_medio):
    """Evalúa si la celda de medio de verificación tiene más de 20 caracteres alfanuméricos o enlaces/soportes."""
    texto = str(texto_medio).strip() if texto_medio else ""
    
    if not texto or texto.lower() in ["nan", "none"]:
        return "No (Falta)"
        
    # Contar cuántos caracteres son letras o números (omitiendo espacios y puntuación)
    caracteres_alfanumericos = len(re.findall(r'[a-zA-Z0-9áéíóúÁÉÍÓÚñÑ]', texto))
    
    # Patrón extra de seguridad (por si tiene pocos caracteres pero es un enlace válido)
    patron_enlace_pdf = r"(?i)(http[s]?://|drive\.google|www\.|\.pdf)"
    
    if caracteres_alfanumericos >= 20 or re.search(patron_enlace_pdf, texto):
        return "Sí (Detectada)"
    else:
        return "No (Insuficiente)"

def evaluar_coherencia_textual(obs_texto, medio_texto):
    """Evalúa si el medio de verificación tiene texto y si comparte palabras clave con la observación."""
    obs = str(obs_texto).strip().lower() if obs_texto else ""
    medio = str(medio_texto).strip().lower() if medio_texto else ""

    if not medio or medio == "nan" or medio == "none":
        return "Sin texto"
        
    if not obs or obs == "nan" or obs == "none":
        return "Observación vacía (No evaluable)"

    # Extraer palabras de 4 o más caracteres alfanuméricos (incluye números)
    patron_palabras = r'\b[a-záéíóúñ0-9]{4,}\b'
    tokens_obs = set(re.findall(patron_palabras, obs))
    tokens_medio = set(re.findall(patron_palabras, medio))

    # Stopwords (palabras comunes a ignorar en el cruce)
    stopwords = {"para", "como", "este", "esta", "pero", "porque", "cuando", "donde", "quien", "desde", "hacia", "sobre", "entre", "hasta", "según", "tiene", "todo", "parte", "cual", "años", "2024", "2025", "2026"}
    
    tokens_obs = tokens_obs - stopwords
    tokens_medio = tokens_medio - stopwords

    # Añadimos \.pdf para que un simple nombre de archivo válido también te dé coherencia directa
    if re.search(r"(http[s]?://|drive\.google|www\.|\.pdf|\.docx|\.xlsx)", medio):
        return "Coherente (Enlace o Archivo detectado)"

    # Calcular intersección de palabras clave
    palabras_comunes = tokens_obs.intersection(tokens_medio)

    # Si comparten al menos 1 palabra clave significativa, se asume relación semántica
    if len(palabras_comunes) > 0:
        return f"Coherente ({len(palabras_comunes)} palabras en común)"
    else:
        return "Incoherente (Sin relación textual)"


def procesar_sabana_observaciones(ruta_excel, ruta_exclusiones=None):
    print("Leyendo Excel (Modo seguro para conservar logos y diseño)...")
    wb = openpyxl.load_workbook(ruta_excel)
    ws = wb.active
    
    nombres_requeridos = ["ENTIDAD", "VIGENCIA", "FECHA_CREACION", "DESCRIPCION_META", "META", "MES", "NUMPROYECTO", "PROYECTO"]
    indices_requeridos = {}
    fila_cabecera = None
    col_obs = None
    
    # 1. Buscar dinámicamente las columnas
    col_medio = None
    for r in range(1, 16):
        for c in range(1, ws.max_column + 1):
            valor_celda = str(ws.cell(row=r, column=c).value).strip().upper()
            
            # Normalizar para quitar tildes y saltos de línea al buscar
            valor_normalizado = valor_celda.replace("\n", " ").replace("Ó", "O")
            
            if valor_celda == "OBSERVACIONES":
                fila_cabecera = r
                col_obs = c
            elif "MEDIO" in valor_normalizado and "VERIFICA" in valor_normalizado:
                col_medio = c
            elif valor_celda in nombres_requeridos:
                indices_requeridos[valor_celda] = c
                
        if fila_cabecera and col_medio is None:
            pass # Sigue buscando si encontró Observaciones pero no Medio Verif
        elif fila_cabecera: 
            break
            
    if not fila_cabecera:
        print("Error: No se encontró la columna OBSERVACIONES.")
        return

    print(f"Cabeceras encontradas en fila {fila_cabecera}. Preparando estructura...")
    
    # 2. Insertar las 5 columnas
    ws.insert_cols(col_obs + 1, amount=5)
    ws.cell(row=fila_cabecera, column=col_obs + 1, value="Cumple_Estructura")
    ws.cell(row=fila_cabecera, column=col_obs + 2, value="Etapa_Extraida")
    ws.cell(row=fila_cabecera, column=col_obs + 3, value="Estado_Extraido")
    ws.cell(row=fila_cabecera, column=col_obs + 4, value="Evidencia_Validada")
    ws.cell(row=fila_cabecera, column=col_obs + 5, value="Coherencia_Medio_Verif")
    
    if col_medio and col_medio > col_obs:
        col_medio += 5
        
    for nombre_col in indices_requeridos:
        if indices_requeridos[nombre_col] > col_obs:
            indices_requeridos[nombre_col] += 5
            
    print("Analizando textos y validando columnas vacías...")
    
    # 3. Iterar y procesar filas
    for row in range(fila_cabecera + 1, ws.max_row + 1):
        fila_incompleta = False
        for nombre_col, col_idx in indices_requeridos.items():
            valor = ws.cell(row=row, column=col_idx).value
            if valor is None or str(valor).strip() == "" or str(valor).strip().lower() == "nan":
                fila_incompleta = True
                break
        
        if fila_incompleta:
            ws.cell(row=row, column=col_obs + 1, value="-")
            ws.cell(row=row, column=col_obs + 2, value="-")
            ws.cell(row=row, column=col_obs + 3, value="-")
            ws.cell(row=row, column=col_obs + 4, value="-")
            ws.cell(row=row, column=col_obs + 5, value="-")
            continue
            
        texto_crudo = ws.cell(row=row, column=col_obs).value
        texto_medio = ws.cell(row=row, column=col_medio).value if col_medio else ""
        
        # 1. Extraer etapa y estado de las Observaciones
        cumple, etapa, estado = analizar_observacion_estricta(texto_crudo)
        
        # 2. Evaluar la cantidad de texto o enlaces en el Medio de Verificación
        evidencia = evaluar_evidencia(texto_medio)
        
        # 3. Evaluar coherencia semántica entre ambas columnas
        coherencia = evaluar_coherencia_textual(texto_crudo, texto_medio) if col_medio else "Columna Medio Verif no hallada"
        
        # Escribir resultados en el Excel
        ws.cell(row=row, column=col_obs + 1, value=cumple)
        ws.cell(row=row, column=col_obs + 2, value=etapa)
        ws.cell(row=row, column=col_obs + 3, value=estado)
        ws.cell(row=row, column=col_obs + 4, value=evidencia)
        ws.cell(row=row, column=col_obs + 5, value=coherencia)
        
    # 4. Guardar base principal procesada
    nueva_ruta = ruta_excel.replace(".xlsx", "_Analizado.xlsx")
    wb.save(nueva_ruta)
    print("Evaluación fila por fila terminada. Generando reportes...")

    # ==========================================
    # 5. CÁLCULO ESTADÍSTICO DE LA META 8 (PANDAS)
    # ==========================================
    df = pd.read_excel(nueva_ruta, header=fila_cabecera - 1)
    
    df_validos = df[df['Cumple_Estructura'] != '-'].copy()

    # Omitir resultados de Secretaría de Gobierno - Nivel Central
    df_validos = df_validos[df_validos['ENTIDAD'].str.strip().str.upper() != 'SECRETARIA DE GOBIERNO - NIVEL CENTRAL']
    
    # NORMALIZAR Y CONVERTIR ENTIDADES AL FORMATO ESTÁNDAR
    df_validos['ENTIDAD'] = df_validos['ENTIDAD'].apply(limpiar_nombre_localidad)

    # -------------------------------------------------------------
    # EXCLUSIÓN DINÁMICA DE METAS (ANTI-JOIN)
    # -------------------------------------------------------------
    if ruta_exclusiones:
        try:
            df_excl = pd.read_excel(ruta_exclusiones)
            
            # Estandarizar nombre de la entidad en las exclusiones
            if 'Entidad' in df_excl.columns:
                df_excl['Entidad'] = df_excl['Entidad'].apply(limpiar_nombre_localidad)
                
            # Renombrar columnas para que coincidan con la sábana principal
            df_excl = df_excl.rename(columns={
                'Año_excluido': 'VIGENCIA',
                'Entidad': 'ENTIDAD',
                'cod_proyecto': 'NUMPROYECTO',
                'cod_meta': 'META'
            })
            
            columnas_cruce = ['VIGENCIA', 'ENTIDAD', 'NUMPROYECTO', 'META']
            
            if all(col in df_excl.columns for col in columnas_cruce):
                # Formatear a texto limpio para evitar que Excel cruce mal números (ej. 2026.0 vs "2026")
                for col in columnas_cruce:
                    df_validos[col] = df_validos[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
                    df_excl[col] = df_excl[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
                
                # Anti-Join: Cruza las bases y elimina de df_validos las filas que hicieron match
                df_validos = df_validos.merge(df_excl[columnas_cruce], on=columnas_cruce, how='left', indicator=True)
                df_validos = df_validos[df_validos['_merge'] == 'left_only'].drop(columns=['_merge'])
                print("Tabla de exclusiones aplicada correctamente.")
            else:
                print("Advertencia: El archivo de exclusiones no contiene las 4 columnas requeridas.")
        except Exception as e:
            print(f"Error al procesar exclusiones: {e}")

    # A. DENOMINADOR
    df_total_metas = df_validos.groupby(['ENTIDAD', 'VIGENCIA', 'MES'])['META'].nunique().reset_index(name='Total_Metas_Diferentes')
    
    # B. NUMERADOR
    df_cumplen = df_validos[(df_validos['Cumple_Estructura'] == 'Sí') & (df_validos['Evidencia_Validada'] == 'Sí (Detectada)')]
    df_metas_cumplen = df_cumplen.groupby(['ENTIDAD', 'VIGENCIA', 'MES'])['META'].nunique().reset_index(name='Metas_Cumplen')
    
    # C. COMBINAR Y CALCULAR PORCENTAJE
    df_resumen = pd.merge(df_total_metas, df_metas_cumplen, on=['ENTIDAD', 'VIGENCIA', 'MES'], how='left')
    df_resumen['Metas_Cumplen'] = df_resumen['Metas_Cumplen'].fillna(0) 
    
    df_resumen['% Avance Meta 8'] = ((df_resumen['Metas_Cumplen'] / df_resumen['Total_Metas_Diferentes']) * 100).round(2)
    df_resumen = df_resumen.sort_values(by=['VIGENCIA', 'MES', 'ENTIDAD'])
    
    # -------------------------------------------------------------
    # D. NUEVO: TABLA CRUZADA (MATRIZ) DE LOCALIDADES POR AÑO/MES
    # -------------------------------------------------------------
    df_matriz = df_resumen.pivot_table(
        index='ENTIDAD',               
        columns=['VIGENCIA', 'MES'],   
        values='% Avance Meta 8'       
    )
    
    # ORDEN DE REFERENCIA EXACTO EN MAYÚSCULAS
    orden_localidades = [
        "01 - USAQUÉN", "02 - CHAPINERO", "03 - SANTA FE", "04 - SAN CRISTÓBAL", 
        "05 - USME", "06 - TUNJUELITO", "07 - BOSA", "08 - KENNEDY", 
        "09 - FONTIBÓN", "10 - ENGATIVÁ", "11 - SUBA", "12 - BARRIOS UNIDOS", 
        "13 - TEUSAQUILLO", "14 - LOS MÁRTIRES", "15 - ANTONIO NARIÑO", 
        "16 - PUENTE ARANDA", "17 - LA CANDELARIA", "18 - RAFAEL URIBE URIBE", 
        "19 - CIUDAD BOLÍVAR", "20 - SUMAPAZ"
    ]
    
    df_matriz = df_matriz.reindex(orden_localidades)

    # E. GUARDAR EN NUEVAS HOJAS
    with pd.ExcelWriter(nueva_ruta, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_resumen.to_excel(writer, sheet_name='Resumen Meta 8', index=False)
        df_matriz.to_excel(writer, sheet_name='Matriz Localidades', startcol=1)

        # --- Magia de Openpyxl para el diseño visual ---
        ws_matriz = writer.sheets['Matriz Localidades']

        # Eliminar la fila 3 (donde Pandas deja la palabra "ENTIDAD" en blanco)
        ws_matriz.delete_rows(3)

        # Identificar hasta qué fila llegan los datos para combinar
        max_fila = ws_matriz.max_row
        
        # Combinar la columna A desde la fila 3 hasta el final
        ws_matriz.merge_cells(start_row=3, start_column=1, end_row=max_fila, end_column=1)
        
        # Escribir "Entidad", centrarlo y rotarlo 90 grados
        celda_entidad = ws_matriz.cell(row=3, column=1, value="Entidad")
        celda_entidad.font = Font(bold=True, size=12)
        celda_entidad.alignment = Alignment(textRotation=90, horizontal='center', vertical='center')
        
        # AUTOAJUSTAR ANCHO DE LAS COLUMNAS (Incluyendo la Columna B)
        max_length = 0
        for cell in ws_matriz["B"]:
            if cell.value:  # Asegura que la celda no esté vacía
                max_length = max(max_length, len(str(cell.value)))
        # 3. Asigna el ancho calculado con el margen de seguridad
        ws_matriz.column_dimensions["B"].width = max(max_length + 3, 12)
        
    print(f"¡Análisis completado con éxito! Archivo guardado en:\n{nueva_ruta}")

# Ejecutar
if __name__ == "__main__":
    ruta_datos = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\Salida_1.xlsx"
    
    # Crea tu archivo Excel con las 4 columnas (Año_excluido, Entidad, cod_proyecto, cod_meta) y pon la ruta aquí
    ruta_excl = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\Metas excluidas.xlsx" 
    
    procesar_sabana_observaciones(ruta_datos, ruta_exclusiones=ruta_excl)
#8m 44s    

Leyendo Excel (Modo seguro para conservar logos y diseño)...
Cabeceras encontradas en fila 1. Preparando estructura...
Analizando textos y validando columnas vacías...
Evaluación fila por fila terminada. Generando reportes...
Tabla de exclusiones aplicada correctamente.
¡Análisis completado con éxito! Archivo guardado en:
C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\Salida_1_Analizado.xlsx


### V1.3

In [7]:
import re
import math
import difflib
import unicodedata
from collections import Counter

import openpyxl
from openpyxl.styles import Alignment, Font
import pandas as pd


# =============================================================
# 1. MOTOR TF-IDF (versión mejorada)
# =============================================================
class MotorRecuperacion:
    def __init__(self, stop_words=None, n_max=3):
        self.stop_words = stop_words if stop_words else set()
        self.n_max = n_max
        self.idf = {}
        self.tf_idf_enciclopedia = []
        self.magnitudes_enciclopedia = []
        self.etiquetas_categorias = []
        self.vocabulario_conocido = set()

    def normalizar(self, texto):
        if not texto:
            return ""
        texto = ''.join(
            c for c in unicodedata.normalize('NFD', str(texto))
            if unicodedata.category(c) != 'Mn'
        )
        texto = texto.lower()
        texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
        texto = re.sub(r'\s+', ' ', texto)
        return texto.strip()

    def corregir_ortografia(self, texto):
        """Corrige sin borrar stopwords (se filtran después)."""
        palabras = self.normalizar(texto).split()
        corregidas = []
        for p in palabras:
            if p in self.vocabulario_conocido or len(p) <= 2:
                corregidas.append(p)
            else:
                matches = difflib.get_close_matches(
                    p, list(self.vocabulario_conocido), n=1, cutoff=0.75
                )
                corregidas.append(matches[0] if matches else p)
        return " ".join(corregidas)

    def obtener_ngramas_combinados(self, texto):
        """Filtra stopwords ANTES de generar n-gramas."""
        palabras = [p for p in texto.split() if p not in self.stop_words]
        if not palabras:
            return []
        ngramas = []
        rango_maximo = min(self.n_max, len(palabras))
        for n in range(1, rango_maximo + 1):
            for i in range(len(palabras) - n + 1):
                ngramas.append(" ".join(palabras[i:i + n]))
        return ngramas

    def cargar_enciclopedia(self, nombres_categorias, documentos_entrenamiento):
        self.etiquetas_categorias = nombres_categorias
        conteo_docs_con_termino = Counter()
        documentos_tokenizados = []

        # Vocabulario conocido (para corrección ortográfica)
        for doc in documentos_entrenamiento:
            for p in self.normalizar(doc).split():
                if p not in self.stop_words and len(p) > 2:
                    self.vocabulario_conocido.add(p)

        # Tokenizar cada documento
        for doc in documentos_entrenamiento:
            texto_limpio = " ".join(
                p for p in self.normalizar(doc).split()
                if p not in self.stop_words
            )
            tokens = self.obtener_ngramas_combinados(texto_limpio)
            documentos_tokenizados.append(tokens)
            for token in set(tokens):
                conteo_docs_con_termino[token] += 1

        N = len(documentos_entrenamiento)
        for token, freq in conteo_docs_con_termino.items():
            self.idf[token] = math.log10((1.0 + N) / freq)

        for tokens in documentos_tokenizados:
            tf_vector = Counter(tokens)
            vector_tfidf = {}
            suma_cuadrados = 0
            for token, count in tf_vector.items():
                tf_w = 1.0 + math.log10(count)
                val_tfidf = tf_w * self.idf[token]
                vector_tfidf[token] = val_tfidf
                suma_cuadrados += val_tfidf ** 2
            self.tf_idf_enciclopedia.append(vector_tfidf)
            self.magnitudes_enciclopedia.append(math.sqrt(suma_cuadrados))

    def clasificar_query(self, texto_query, umbral_similitud=0.18, margen_ambiguo=0.05):
        texto_corregido = self.corregir_ortografia(texto_query)
        tokens_query = self.obtener_ngramas_combinados(texto_corregido)

        if not tokens_query:
            return "-", 0.0, "Sin palabras válidas"

        tf_query = Counter(tokens_query)
        vector_query = {}
        suma_cuadrados_query = 0
        for token, count in tf_query.items():
            if token in self.idf:
                tf_w = 1.0 + math.log10(count)
                val_tfidf = tf_w * self.idf[token]
                vector_query[token] = val_tfidf
                suma_cuadrados_query += val_tfidf ** 2

        mag_query = math.sqrt(suma_cuadrados_query)
        if mag_query == 0:
            return "-", 0.0, "Coseno = 0"

        resultados = []
        for idx, vec_cat in enumerate(self.tf_idf_enciclopedia):
            producto_punto = sum(
                vector_query.get(t, 0) * vec_cat.get(t, 0) for t in vec_cat
            )
            mag_cat = self.magnitudes_enciclopedia[idx]
            similitud = producto_punto / (mag_query * mag_cat) if mag_cat > 0 else 0
            resultados.append((self.etiquetas_categorias[idx], similitud))

        resultados.sort(key=lambda x: x[1], reverse=True)
        mejor_cat, mejor_score = resultados[0]

        debug_str = " | ".join(
            f"{cat}: {score:.3f}" for cat, score in resultados if score > 0
        ) or "0.0 en todas"

        if mejor_score < umbral_similitud:
            return "-", mejor_score, f"Bajo umbral ({debug_str})"

        if len(resultados) > 1:
            segundo_score = resultados[1][1]
            if (mejor_score - segundo_score) < margen_ambiguo:
                return mejor_cat, mejor_score, f"Ambiguo (2º={segundo_score:.3f})"

        return mejor_cat, mejor_score, f"OK ({debug_str})"


# =============================================================
# 2. MAPEO SIPSE (patrones SIN tildes, ya normalizados)
# =============================================================
MAPEO_SIPSE = [
    {
        "estado": "Actualización de criterios",
        "etapa": "Proyecto de Inversión",
        "peso_min": 0.20,
        "variantes": [
            r"actualizacion de criterios de elegibilidad y viabilidad",
            r"actualizacion de criterios",
            r"criterios de elegibilidad",
            r"criterios de viabilidad",
        ],
    },
    {
        "estado": "Asistencia técnica del sector",
        "etapa": "Proyecto de Inversión",
        "peso_min": 0.20,
        "variantes": [
            r"asistencia tecnica del sector",
            r"asistencia tecnica",
            r"apoyo tecnico del sector",
        ],
    },
    {
        "estado": "Cocreación de propuestas",
        "etapa": "Proyecto de Inversión",
        "peso_min": 0.20,
        "variantes": [
            r"cocreacion de propuestas",
            r"cocreacion",
            r"construccion conjunta de propuestas",
        ],
    },
    {
        "estado": "Actualización DTS/Ficha EBI",
        "etapa": "Proyecto de Inversión",
        "peso_min": 0.20,
        "variantes": [
            r"actualizacion documento tecnico de soporte",
            r"actualizacion dts",
            r"actualizacion ficha ebi",
            r"documento tecnico de soporte",
        ],
    },
    # ---------- PRECONTRACTUAL ----------
    {
        "estado": "Estudios Previos y anexos",
        "etapa": "Precontractual",
        "peso_min": 0.20,
        "variantes": [
            r"elaboracion de estudios previos",
            r"estudios previos",
            r"documentos anexos",
        ],
    },
    {
        "estado": "Comité de Contratación",
        "etapa": "Precontractual",
        "peso_min": 0.22,
        "variantes": [
            r"comite de contratacion",
            r"sesion del comite",
            r"reunion del comite",
            r"comite de contratacion",
        ],
    },
    {
        "estado": "Asistencia Técnica (Central)",
        "etapa": "Precontractual",
        "peso_min": 0.22,
        "variantes": [
            r"asistencia tecnica \(nivel central\)",
            r"asistencia tecnica nivel central",
            r"asistencia tecnica central",
        ],
    },
    {
        "estado": "Publicación proceso",
        "etapa": "Precontractual",
        "peso_min": 0.22,
        "variantes": [
            r"publicacion proceso contractual",
            r"publicacion del proceso",
            r"proceso publicado",
        ],
    },
    {
        "estado": "Adjudicación y suscripción",
        "etapa": "Precontractual",
        "peso_min": 0.20,
        "variantes": [
            r"adjudicacion y suscripcion de proceso contractual",
            r"adjudicacion y suscripcion",
            r"adjudicacion del contrato",
            r"suscripcion del contrato",
        ],
    },
    # ---------- EJECUCIÓN CONTRACTUAL ----------
    # IMPORTANTE: este bloque va ANTES que cualquier otro que pueda
    # contener "avance" o "%", para ganar en caso de empate.
    {
        "estado": "En ejecución",
        "etapa": "Ejecución Contractual",
        "peso_min": 0.22,
        "variantes": [
            # Frases explícitas
            r"en ejecucion del contrato",
            r"en ejecucion",
            r"en la ejecucion",
            r"ejecutando el contrato",
            r"contrato en ejecucion",
            r"contrato en curso",
            r"en desarrollo del contrato",
            r"ejecucion contractual",
            # Patrones estructurales: % + contexto de ejecución
            r"(?:ejecuc|ejecutad|contrato).{0,40}?\d{1,3}\s*(?:%|por\s*ciento)",
            r"\d{1,3}\s*(?:%|por\s*ciento).{0,40}?(?:ejecuc|ejecutad|avance)",
            # % suelto tras frase de ejecución
            r"en ejecucion.{0,30}?avance",
        ],
    },
    {
        "estado": "Inicio ejecución contrato",
        "etapa": "Ejecución Contractual",
        "peso_min": 0.22,
        "variantes": [
            r"inicio ejecucion del contrato",
            r"inicio de ejecucion",
            r"contrato iniciado",
            r"se dio inicio al contrato",
        ],
    },
    {
        "estado": "En Modificación contractual",
        "etapa": "Ejecución Contractual",
        "peso_min": 0.22,
        "variantes": [
            r"en modificacion contractual",
            r"modificacion contractual",
            r"modificacion al contrato",
            r"adicion al contrato",
            r"otrosi",
        ],
    },
    # ---------- POSTCONTRACTUAL ----------
    {
        "estado": "Finalización ejecución",
        "etapa": "Postcontractual",
        "peso_min": 0.22,
        "variantes": [
            r"finalizacion ejecucion contrato",
            r"finalizacion de la ejecucion",
            r"contrato finalizado",
            r"ejecucion finalizada",
        ],
    },
    {
        "estado": "En proceso de liquidación",
        "etapa": "Postcontractual",
        "peso_min": 0.22,
        "variantes": [
            r"en proceso de liquidacion",
            r"proceso de liquidacion",
            r"liquidacion del contrato",
            r"contrato en liquidacion",
        ],
    },
]

PREFIJO_ETAPA = re.compile(
    r'^\s*(?:etapa|fase|num|n[°º]|item|paso)?\s*[\d]{1,2}\s*[\.\)\-:]?\s*',
    re.IGNORECASE
)


# =============================================================
# 3. FUNCIONES DE ANÁLISIS
# =============================================================
def _normalizar_texto(t):
    if not t:
        return ""
    t = unicodedata.normalize('NFKD', str(t))
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = t.lower()
    t = re.sub(r'\s+', ' ', t)
    return t.strip()


def analizar_observacion_estricta(texto_crudo, motor_recuperacion=None):
    """
    Retorna (cumple, etapa, estado).
    cumple: 'Sí' | 'Sí (por TF-IDF)' | 'No'
    """
    texto = _normalizar_texto(texto_crudo)
    if not texto:
        return "No", "Sin Etapa", "Sin Estado"

    texto_sin_prefijo = PREFIJO_ETAPA.sub('', texto, count=1)

    # Capa 1: variantes explícitas + patrones estructurales
    for bloque in MAPEO_SIPSE:
        for variante in bloque["variantes"]:
            if re.search(variante, texto_sin_prefijo):
                return "Sí", bloque["etapa"], bloque["estado"]

    # Capa 2: fallback TF-IDF
    if motor_recuperacion is not None:
        categoria, score, _debug = motor_recuperacion.clasificar_query(texto_crudo)
        if categoria != "-":
            for bloque in MAPEO_SIPSE:
                if bloque["estado"] == categoria and score >= bloque["peso_min"]:
                    return "Sí (por TF-IDF)", bloque["etapa"], bloque["estado"]

    return "No", "Sin Etapa", "Sin Estado"


def limpiar_nombre_localidad(entidad):
    """Estandariza el nombre de la localidad en mayúsculas exactas."""
    if not isinstance(entidad, str):
        return ""
    entidad_str = entidad.strip().upper()
    nfkd = unicodedata.normalize('NFKD', entidad_str)
    entidad_limpia = "".join([c for c in nfkd if not unicodedata.combining(c)])

    if "USAQUEN" in entidad_limpia: return "01 - USAQUÉN"
    if "CHAPINERO" in entidad_limpia: return "02 - CHAPINERO"
    if "SANTA FE" in entidad_limpia: return "03 - SANTA FE"
    if "SAN CRISTOBAL" in entidad_limpia: return "04 - SAN CRISTÓBAL"
    if "USME" in entidad_limpia: return "05 - USME"
    if "TUNJUELITO" in entidad_limpia: return "06 - TUNJUELITO"
    if "BOSA" in entidad_limpia: return "07 - BOSA"
    if "KENNEDY" in entidad_limpia: return "08 - KENNEDY"
    if "FONTIBON" in entidad_limpia: return "09 - FONTIBÓN"
    if "ENGATIVA" in entidad_limpia: return "10 - ENGATIVÁ"
    if "SUBA" in entidad_limpia: return "11 - SUBA"
    if "BARRIOS UNIDOS" in entidad_limpia: return "12 - BARRIOS UNIDOS"
    if "TEUSAQUILLO" in entidad_limpia: return "13 - TEUSAQUILLO"
    if "MARTIRES" in entidad_limpia: return "14 - LOS MÁRTIRES"
    if "ANTONIO NARINO" in entidad_limpia: return "15 - ANTONIO NARIÑO"
    if "PUENTE ARANDA" in entidad_limpia: return "16 - PUENTE ARANDA"
    if "CANDELARIA" in entidad_limpia: return "17 - LA CANDELARIA"
    if "RAFAEL URIBE" in entidad_limpia: return "18 - RAFAEL URIBE URIBE"
    if "CIUDAD BOLIVAR" in entidad_limpia: return "19 - CIUDAD BOLÍVAR"
    if "SUMAPAZ" in entidad_limpia: return "20 - SUMAPAZ"

    return entidad_str


def evaluar_evidencia(texto_medio):
    """Valida si la celda de medio de verificación tiene contenido sustancial."""
    texto = str(texto_medio).strip() if texto_medio else ""
    if not texto or texto.lower() in ["nan", "none"]:
        return "No (Falta)"

    caracteres_alfanumericos = len(
        re.findall(r'[a-zA-Z0-9áéíóúÁÉÍÓÚñÑ]', texto)
    )
    patron_enlace_pdf = r"(?i)(http[s]?://|drive\.google|www\.|\.pdf)"

    if caracteres_alfanumericos >= 20 or re.search(patron_enlace_pdf, texto):
        return "Sí (Detectada)"
    return "No (Insuficiente)"


def evaluar_coherencia_textual(obs_texto, medio_texto):
    """Evalúa si el medio comparte palabras clave con la observación."""
    obs = str(obs_texto).strip().lower() if obs_texto else ""
    medio = str(medio_texto).strip().lower() if medio_texto else ""

    if not medio or medio in ("nan", "none"):
        return "Sin texto"
    if not obs or obs in ("nan", "none"):
        return "Observación vacía (No evaluable)"

    if re.search(r"(http[s]?://|drive\.google|www\.|\.pdf|\.docx|\.xlsx)", medio):
        return "Coherente (Enlace o Archivo detectado)"

    patron_palabras = r'\b[a-záéíóúñ0-9]{4,}\b'
    tokens_obs = set(re.findall(patron_palabras, obs))
    tokens_medio = set(re.findall(patron_palabras, medio))

    stopwords = {
        "para", "como", "este", "esta", "pero", "porque", "cuando",
        "donde", "quien", "desde", "hacia", "sobre", "entre", "hasta",
        "según", "tiene", "todo", "parte", "cual", "años",
        "2024", "2025", "2026",
    }
    tokens_obs -= stopwords
    tokens_medio -= stopwords

    palabras_comunes = tokens_obs.intersection(tokens_medio)
    if len(palabras_comunes) > 0:
        return f"Coherente ({len(palabras_comunes)} palabras en común)"
    return "Incoherente (Sin relación textual)"


# =============================================================
# 4. CARGA DE STOPWORDS Y ENCICLOPEDIA
# =============================================================
def cargar_stopwords(ruta_excel, columna="Stopword"):
    """Carga stopwords desde un Excel (1 columna)."""
    try:
        df = pd.read_excel(ruta_excel)
        col = next((c for c in df.columns if c.lower() == columna.lower()), df.columns[0])
        sw = set(
            str(x).strip().lower()
            for x in df[col].dropna().tolist()
            if str(x).strip()
        )
        print(f"Stopwords cargadas: {len(sw)}")
        return sw
    except Exception as e:
        print(f"[!] No se pudieron cargar stopwords: {e}")
        return set()


def cargar_enciclopedia(ruta_excel, col_categoria="Categoria", col_texto="Texto"):
    """Carga la enciclopedia: lista de (categoria, texto_ejemplo)."""
    try:
        df = pd.read_excel(ruta_excel)
        # Normalizar nombres de columnas
        cols = {c.lower(): c for c in df.columns}
        cat_col = next((cols[c] for c in cols if "categ" in c), df.columns[0])
        txt_col = next((cols[c] for c in cols if "text" in c or "ejem" in c or "patron" in c),
                       df.columns[1] if len(df.columns) > 1 else df.columns[0])

        etiquetas = df[cat_col].astype(str).str.strip().tolist()
        textos = df[txt_col].astype(str).str.strip().tolist()
        print(f"Enciclopedia cargada: {len(etiquetas)} documentos")
        return etiquetas, textos
    except Exception as e:
        print(f"[!] No se pudo cargar enciclopedia: {e}")
        return [], []


# =============================================================
# 5. PROCESO PRINCIPAL
# =============================================================
def procesar_sabana_observaciones(ruta_excel, ruta_exclusiones=None,
                                  ruta_stopwords=None, ruta_enciclopedia=None):
    print("Leyendo Excel (Modo seguro para conservar logos y diseño)...")
    wb = openpyxl.load_workbook(ruta_excel)
    ws = wb.active

    # --- Cargar motor TF-IDF ---
    motor = None
    if ruta_enciclopedia:
        stopwords = cargar_stopwords(ruta_stopwords) if ruta_stopwords else set()
        etiquetas, textos = cargar_enciclopedia(ruta_enciclopedia)
        if etiquetas and textos:
            motor = MotorRecuperacion(stop_words=stopwords, n_max=3)
            motor.cargar_enciclopedia(etiquetas, textos)
            print("Motor TF-IDF inicializado correctamente.")
        else:
            print("[!] Enciclopedia vacía; se omite fallback TF-IDF.")

    nombres_requeridos = ["ENTIDAD", "VIGENCIA", "FECHA_CREACION", "DESCRIPCION_META",
                          "META", "MES", "NUMPROYECTO", "PROYECTO"]
    indices_requeridos = {}
    fila_cabecera = None
    col_obs = None
    col_medio = None

    # 1. Buscar dinámicamente las columnas
    for r in range(1, 16):
        for c in range(1, ws.max_column + 1):
            valor_celda = str(ws.cell(row=r, column=c).value).strip().upper()
            valor_normalizado = valor_celda.replace("\n", " ").replace("Ó", "O")

            if valor_celda == "OBSERVACIONES":
                fila_cabecera = r
                col_obs = c
            elif "MEDIO" in valor_normalizado and "VERIFICA" in valor_normalizado:
                col_medio = c
            elif valor_celda in nombres_requeridos:
                indices_requeridos[valor_celda] = c

        if fila_cabecera and col_medio is None:
            pass
        elif fila_cabecera:
            break

    if not fila_cabecera:
        print("Error: No se encontró la columna OBSERVACIONES.")
        return

    print(f"Cabeceras encontradas en fila {fila_cabecera}. Preparando estructura...")

    # 2. Insertar las 5 columnas
    ws.insert_cols(col_obs + 1, amount=5)
    ws.cell(row=fila_cabecera, column=col_obs + 1, value="Cumple_Estructura")
    ws.cell(row=fila_cabecera, column=col_obs + 2, value="Etapa_Extraida")
    ws.cell(row=fila_cabecera, column=col_obs + 3, value="Estado_Extraido")
    ws.cell(row=fila_cabecera, column=col_obs + 4, value="Evidencia_Validada")
    ws.cell(row=fila_cabecera, column=col_obs + 5, value="Coherencia_Medio_Verif")

    if col_medio and col_medio > col_obs:
        col_medio += 5

    for nombre_col in indices_requeridos:
        if indices_requeridos[nombre_col] > col_obs:
            indices_requeridos[nombre_col] += 5

    print("Analizando textos y validando columnas vacías...")

    # 3. Iterar y procesar filas
    for row in range(fila_cabecera + 1, ws.max_row + 1):
        fila_incompleta = False
        for _nombre_col, col_idx in indices_requeridos.items():
            valor = ws.cell(row=row, column=col_idx).value
            if valor is None or str(valor).strip() == "" or str(valor).strip().lower() == "nan":
                fila_incompleta = True
                break

        if fila_incompleta:
            for offset in range(1, 6):
                ws.cell(row=row, column=col_obs + offset, value="-")
            continue

        texto_crudo = ws.cell(row=row, column=col_obs).value
        texto_medio = ws.cell(row=row, column=col_medio).value if col_medio else ""

        cumple, etapa, estado = analizar_observacion_estricta(texto_crudo, motor_recuperacion=motor)
        evidencia = evaluar_evidencia(texto_medio)
        coherencia = (
            evaluar_coherencia_textual(texto_crudo, texto_medio)
            if col_medio else "Columna Medio Verif no hallada"
        )

        ws.cell(row=row, column=col_obs + 1, value=cumple)
        ws.cell(row=row, column=col_obs + 2, value=etapa)
        ws.cell(row=row, column=col_obs + 3, value=estado)
        ws.cell(row=row, column=col_obs + 4, value=evidencia)
        ws.cell(row=row, column=col_obs + 5, value=coherencia)

    # 4. Guardar base procesada
    nueva_ruta = ruta_excel.replace(".xlsx", "_AnalizadoV1_3.xlsx")
    wb.save(nueva_ruta)
    print("Evaluación fila por fila terminada. Generando reportes...")

    # ==========================================
    # 5. CÁLCULO ESTADÍSTICO (PANDAS)
    # ==========================================
    df = pd.read_excel(nueva_ruta, header=fila_cabecera - 1)
    df_validos = df[df['Cumple_Estructura'] != '-'].copy()

    df_validos = df_validos[
        df_validos['ENTIDAD'].str.strip().str.upper() != 'SECRETARIA DE GOBIERNO - NIVEL CENTRAL'
    ]
    df_validos['ENTIDAD'] = df_validos['ENTIDAD'].apply(limpiar_nombre_localidad)

    # --- Exclusiones dinámicas ---
    if ruta_exclusiones:
        try:
            df_excl = pd.read_excel(ruta_exclusiones)
            if 'Entidad' in df_excl.columns:
                df_excl['Entidad'] = df_excl['Entidad'].apply(limpiar_nombre_localidad)

            df_excl = df_excl.rename(columns={
                'Año_excluido': 'VIGENCIA',
                'Entidad': 'ENTIDAD',
                'cod_proyecto': 'NUMPROYECTO',
                'cod_meta': 'META',
            })

            columnas_cruce = ['VIGENCIA', 'ENTIDAD', 'NUMPROYECTO', 'META']
            if all(col in df_excl.columns for col in columnas_cruce):
                for col in columnas_cruce:
                    df_validos[col] = (df_validos[col].astype(str).str.strip()
                                       .str.replace(r'\.0$', '', regex=True))
                    df_excl[col] = (df_excl[col].astype(str).str.strip()
                                    .str.replace(r'\.0$', '', regex=True))

                df_validos = df_validos.merge(
                    df_excl[columnas_cruce], on=columnas_cruce,
                    how='left', indicator=True
                )
                df_validos = df_validos[df_validos['_merge'] == 'left_only'].drop(columns=['_merge'])
                print("Tabla de exclusiones aplicada correctamente.")
            else:
                print("Advertencia: El archivo de exclusiones no contiene las 4 columnas requeridas.")
        except Exception as e:
            print(f"Error al procesar exclusiones: {e}")

    # A. Denominador
    df_total_metas = (df_validos.groupby(['ENTIDAD', 'VIGENCIA', 'MES'])['META']
                      .nunique().reset_index(name='Total_Metas_Diferentes'))

    # B. Numerador: cuenta tanto match estricto como TF-IDF
    df_cumplen = df_validos[
        df_validos['Cumple_Estructura'].isin(['Sí', 'Sí (por TF-IDF)'])
        & (df_validos['Evidencia_Validada'] == 'Sí (Detectada)')
    ]
    df_metas_cumplen = (df_cumplen.groupby(['ENTIDAD', 'VIGENCIA', 'MES'])['META']
                        .nunique().reset_index(name='Metas_Cumplen'))

    # C. Combinar y calcular porcentaje
    df_resumen = pd.merge(df_total_metas, df_metas_cumplen,
                          on=['ENTIDAD', 'VIGENCIA', 'MES'], how='left')
    df_resumen['Metas_Cumplen'] = df_resumen['Metas_Cumplen'].fillna(0)
    df_resumen['% Avance Meta 8'] = (
        (df_resumen['Metas_Cumplen'] / df_resumen['Total_Metas_Diferentes']) * 100
    ).round(2)
    df_resumen = df_resumen.sort_values(by=['VIGENCIA', 'MES', 'ENTIDAD'])

    # D. Matriz cruzada
    df_matriz = df_resumen.pivot_table(
        index='ENTIDAD', columns=['VIGENCIA', 'MES'], values='% Avance Meta 8'
    )

    orden_localidades = [
        "01 - USAQUÉN", "02 - CHAPINERO", "03 - SANTA FE", "04 - SAN CRISTÓBAL",
        "05 - USME", "06 - TUNJUELITO", "07 - BOSA", "08 - KENNEDY",
        "09 - FONTIBÓN", "10 - ENGATIVÁ", "11 - SUBA", "12 - BARRIOS UNIDOS",
        "13 - TEUSAQUILLO", "14 - LOS MÁRTIRES", "15 - ANTONIO NARIÑO",
        "16 - PUENTE ARANDA", "17 - LA CANDELARIA", "18 - RAFAEL URIBE URIBE",
        "19 - CIUDAD BOLÍVAR", "20 - SUMAPAZ",
    ]
    df_matriz = df_matriz.reindex(orden_localidades)

    # E. Guardar hojas adicionales
    with pd.ExcelWriter(nueva_ruta, engine='openpyxl', mode='a',
                        if_sheet_exists='replace') as writer:
        df_resumen.to_excel(writer, sheet_name='Resumen Meta 8', index=False)
        df_matriz.to_excel(writer, sheet_name='Matriz Localidades', startcol=1)

        ws_matriz = writer.sheets['Matriz Localidades']
        ws_matriz.delete_rows(3)

        max_fila = ws_matriz.max_row
        ws_matriz.merge_cells(start_row=3, start_column=1,
                              end_row=max_fila, end_column=1)

        celda_entidad = ws_matriz.cell(row=3, column=1, value="Entidad")
        celda_entidad.font = Font(bold=True, size=12)
        celda_entidad.alignment = Alignment(textRotation=90,
                                            horizontal='center', vertical='center')

        max_length = 0
        for cell in ws_matriz["B"]:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        ws_matriz.column_dimensions["B"].width = max(max_length + 3, 12)

    print(f"¡Análisis completado con éxito! Archivo guardado en:\n{nueva_ruta}")


# =============================================================
# 6. EJECUCIÓN
# =============================================================
if __name__ == "__main__":
    ruta_datos = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\Salida_1.xlsx"
    ruta_excl  = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\Metas excluidas.xlsx"
    ruta_sw    = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\stopwords.xlsx"
    ruta_enc   = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\enciclopedia.xlsx"

    procesar_sabana_observaciones(
        ruta_datos,
        ruta_exclusiones=ruta_excl,
        ruta_stopwords=ruta_sw,
        ruta_enciclopedia=ruta_enc,
    )

Leyendo Excel (Modo seguro para conservar logos y diseño)...
[!] No se pudieron cargar stopwords: [Errno 2] No such file or directory: 'C:\\Users\\Pc\\OneDrive\\Trabajo\\2026\\Secretaria de Gobierno\\08 Septiembre\\ObservacionesMeta8\\stopwords.xlsx'
[!] No se pudo cargar enciclopedia: [Errno 2] No such file or directory: 'C:\\Users\\Pc\\OneDrive\\Trabajo\\2026\\Secretaria de Gobierno\\08 Septiembre\\ObservacionesMeta8\\enciclopedia.xlsx'
[!] Enciclopedia vacía; se omite fallback TF-IDF.
Cabeceras encontradas en fila 1. Preparando estructura...
Analizando textos y validando columnas vacías...
Evaluación fila por fila terminada. Generando reportes...
Tabla de exclusiones aplicada correctamente.
¡Análisis completado con éxito! Archivo guardado en:
C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\Salida_1_AnalizadoV1_3.xlsx


### V1.4 Claude

In [ ]:
# -*- coding: utf-8 -*-
"""
Evaluación Meta 8 - Observaciones SIPSE (v2)

Secciones:
  0. Configuración (rutas y nombres de columnas)
  1. Utilidades generales (normalización, tkinter, lectura de Excel)
  2. Motor TF-IDF + mapeo de estructura SIPSE
  3. SECCIÓN A: metasExcluidas{AÑO}.txt   (metas con magnitud vacía, "-" o 0)
  4. SECCIÓN B: metasAReportar{AÑO}.txt   (denominador: metas que debe reportar cada localidad por mes)
  5. Reglas de calificación por año (aquí se agregan 2027, 2028, ...)
  6. Proceso principal y reportes
"""

import re
import math
import difflib
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import openpyxl
from openpyxl.styles import Alignment, Font
import pandas as pd

# =============================================================
# 0. CONFIGURACIÓN
# =============================================================
# Convención de rutas:
#   ""   -> se pide el archivo con un explorador (tkinter)
#   None -> no se usa (solo para archivos opcionales)
CARPETA_BASE = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8"

CONFIG = {
    "ruta_datos": rf"{CARPETA_BASE}\Salida_1.xlsx",      # Sábana de observaciones SIPSE
    "carpeta_txt": CARPETA_BASE,                          # Donde viven/se crean los .txt
    "ruta_stopwords": rf"{CARPETA_BASE}\stopwords.xlsx",  # Opcional (TF-IDF)
    "ruta_enciclopedia": rf"{CARPETA_BASE}\enciclopedia.xlsx",  # Opcional (TF-IDF)
    # Archivos fuente para construir los .txt cuando no existan (por año):
    "rutas_plan_metas": {2026: ""},        # Excel con "Magnitud Meta anualizada DEFINITIVA {año}"
    "rutas_consulta_sipse": {2026: ""},    # Excel "consulta avance de metas" de SIPSE
    # True = vuelve a construir los .txt aunque ya existan (úselo cuando cambie el Excel de metas).
    "regenerar_txt": False,
}

# A partir de este año existen los archivos de metas excluidas / metas a reportar.
AÑO_INICIO_PLAN = 2026

# Similitud TF-IDF (0-1) por debajo de la cual el .txt marca la meta para revisión.
# Nunca impide la exclusión: toda meta excluida recibe siempre una meta de SIPSE.
UMBRAL_AVISO_SIMILITUD = 0.40

# Si el Excel de metas viene repartido en varias hojas con la misma cabecera, se leen TODAS.
LEER_TODAS_LAS_HOJAS_PLAN = True

# --- Nombres de columnas (se aceptan varios candidatos; la búsqueda ignora tildes/mayúsculas) ---
COLS_PLAN = {
    "localidad": ["Cod. Localidad", "Codigo Localidad", "Cod Localidad"],
    "nombre_meta": ["Meta proyecto 2025-2028 (PDL)"],
    "cod_meta": ["Cod. Meta", "Codigo Meta", "Cod Meta", "No. Meta", "Numero Meta", "Meta No."],
}

def col_magnitud(año):
    return [f"Magnitud Meta anualizada DEFINITIVA {año}"]

COLS_SIPSE = {
    "localidad": ["Cod. Localidad", "ENTIDAD", "Localidad"],
    "cod_meta": ["META", "Cod. Meta", "Codigo Meta"],
    "nombre_meta": ["DESCRIPCION_META", "Descripcion Meta", "Nombre Meta"],
    "vigencia": ["VIGENCIA", "Vigencia", "Año"],
    "proyecto": ["NUMPROYECTO", "Cod. Proyecto", "Codigo Proyecto"],  # opcional
}

COLS_REQUERIDAS_DATOS = ["ENTIDAD", "VIGENCIA", "FECHA_CREACION", "DESCRIPCION_META",
                         "META", "MES", "NUMPROYECTO", "PROYECTO"]

# Rótulos originales del formato (no cambiar)
NUEVAS_COLUMNAS = ["Cumple_Estructura", "Etapa_Extraida", "Estado_Extraido",
                   "Evidencia_Validada", "Coherencia_Medio_Verif"]
SUFIJO_SALIDA = "_AnalizadoV1_4.xlsx"


# =============================================================
# 1. UTILIDADES GENERALES
# =============================================================
VACIOS = {"", "nan", "none", "nat"}
VACIOS_O_GUION = VACIOS | {"-", "–", "—"}

LOCALIDADES = [
    ("USAQUEN", "01 - USAQUÉN"), ("CHAPINERO", "02 - CHAPINERO"), ("SANTA FE", "03 - SANTA FE"),
    ("SAN CRISTOBAL", "04 - SAN CRISTÓBAL"), ("USME", "05 - USME"), ("TUNJUELITO", "06 - TUNJUELITO"),
    ("BOSA", "07 - BOSA"), ("KENNEDY", "08 - KENNEDY"), ("FONTIBON", "09 - FONTIBÓN"),
    ("ENGATIVA", "10 - ENGATIVÁ"), ("SUBA", "11 - SUBA"), ("BARRIOS UNIDOS", "12 - BARRIOS UNIDOS"),
    ("TEUSAQUILLO", "13 - TEUSAQUILLO"), ("MARTIRES", "14 - LOS MÁRTIRES"),
    ("ANTONIO NARINO", "15 - ANTONIO NARIÑO"), ("PUENTE ARANDA", "16 - PUENTE ARANDA"),
    ("CANDELARIA", "17 - LA CANDELARIA"), ("RAFAEL URIBE", "18 - RAFAEL URIBE URIBE"),
    ("CIUDAD BOLIVAR", "19 - CIUDAD BOLÍVAR"), ("SUMAPAZ", "20 - SUMAPAZ"),
]
ETIQUETA_POR_CODIGO = {i + 1: etiqueta for i, (_, etiqueta) in enumerate(LOCALIDADES)}

MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7,
         "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}


def quitar_tildes(texto):
    nfkd = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def es_vacio(valor, conjunto=VACIOS):
    if valor is None:
        return True
    if isinstance(valor, float) and math.isnan(valor):
        return True
    return str(valor).strip().lower() in conjunto


def a_numero(valor):
    """Convierte 0, '0', ' 0,00 ', '1.234,5' a float; None si no es número."""
    if isinstance(valor, bool):
        return None
    if isinstance(valor, (int, float)):
        return None if (isinstance(valor, float) and math.isnan(valor)) else float(valor)
    s = str(valor).strip().replace(" ", "")
    if re.fullmatch(r"-?\d{1,3}(\.\d{3})+(,\d+)?", s):   # 1.234,5
        s = s.replace(".", "").replace(",", ".")
    else:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def motivo_exclusion_magnitud(valor):
    """Motivo por el que la magnitud anualizada excluye la meta, o None si la meta debe reportarse.
    Ojo: en Excel el formato contable muestra '-' cuando el valor real es 0."""
    if es_vacio(valor):
        return "vacía"
    if str(valor).strip() in {"-", "–", "—"}:
        return "guion"
    n = a_numero(valor)
    if n is not None and abs(n) < 1e-9:
        return "cero (se ve '-')"
    return None


def a_entero(valor):
    try:
        return int(float(str(valor).strip()))
    except (TypeError, ValueError):
        return None


def normalizar_codigo(valor):
    """'0012', 12, 12.0, '12.0' -> '12'."""
    if es_vacio(valor):
        return ""
    if isinstance(valor, float) and valor.is_integer():
        valor = int(valor)
    s = re.sub(r"\.0+$", "", str(valor).strip())
    return s.lstrip("0") or "0"


def limpiar_nombre_localidad(entidad):
    """Estandariza el nombre de la localidad ('01 - USAQUÉN'). Si no la reconoce, devuelve el texto en mayúsculas."""
    if not isinstance(entidad, str):
        return ""
    limpio = quitar_tildes(entidad.strip().upper())
    for clave, etiqueta in LOCALIDADES:
        if clave in limpio:
            return etiqueta
    return entidad.strip().upper()


def codigo_localidad(valor):
    """Convierte 1, '01', '1.0', '01 - USAQUÉN' o 'Alcaldía Local de Usaquén' en el entero 1..20 (None si no aplica)."""
    if es_vacio(valor):
        return None
    if isinstance(valor, (int, float)):
        n = int(valor)
        return n if 1 <= n <= 20 else None
    s = str(valor).strip()
    m = re.match(r"^0*(\d{1,2})(?:\.0+)?(?!\d)", s)  # acepta '1', '01', '001', '1.0'
    if m:
        n = int(m.group(1))
        return n if 1 <= n <= 20 else None
    m = re.match(r"^(\d{2}) - ", limpiar_nombre_localidad(s))
    return int(m.group(1)) if m else None


def orden_mes(mes):
    n = a_entero(mes)
    if n is not None:
        return n
    return MESES.get(quitar_tildes(str(mes)).strip().lower(), 99)


def _norm_columna(nombre):
    t = quitar_tildes(str(nombre)).lower()
    t = re.sub(r"[^a-z0-9]+", " ", t)
    return t.strip()


def buscar_columna(columnas, candidatos, obligatoria=True, contexto=""):
    """Busca una columna por coincidencia exacta normalizada; si no, por 'contiene'."""
    mapa = {_norm_columna(c): c for c in columnas}
    cands = [_norm_columna(c) for c in candidatos]
    for c in cands:
        if c in mapa:
            return mapa[c]
    for c in cands:
        if len(c) > 6:
            for norm, original in mapa.items():
                if c in norm:
                    return original
    if obligatoria:
        raise KeyError(f"No se encontró la columna {candidatos} {contexto}. "
                       f"Columnas disponibles: {list(columnas)}")
    return None


def _valor_mostrado(celda):
    """Valor de la celda tal como se VE en Excel en el caso del guion contable:
    un 0 con formato contable/personalizado que muestra "-" se devuelve como "-"."""
    v = celda.value
    if isinstance(v, (int, float)) and not isinstance(v, bool) and v == 0:
        fmt = celda.number_format or ""
        if '"-"' in fmt or "\\-" in fmt:
            return "-"
    return v


def _filas_hoja_openpyxl(ws):
    """Lee una hoja propagando el valor de las celdas COMBINADAS a todas las celdas del rango
    (pandas solo deja el valor en la primera fila; el resto queda vacío y se perdía)."""
    filas = [[_valor_mostrado(c) for c in fila] for fila in ws.iter_rows()]
    for rango in ws.merged_cells.ranges:
        if rango.min_row - 1 >= len(filas):
            continue
        valor = filas[rango.min_row - 1][rango.min_col - 1]
        for r in range(rango.min_row - 1, min(rango.max_row, len(filas))):
            for c in range(rango.min_col - 1, min(rango.max_col, len(filas[r]))):
                filas[r][c] = valor
    return filas


def _columnas_unicas(nombres):
    vistos, salida = Counter(), []
    for n in nombres:
        n = re.sub(r"\s+", " ", str(n)).strip() if not es_vacio(n) else "Columna"
        salida.append(n if vistos[n] == 0 else f"{n}.{vistos[n]}")
        vistos[n] += 1
    return salida


def _hojas_como_filas(ruta):
    """Devuelve [(nombre_hoja, filas)] usando openpyxl (xlsx/xlsm) o pandas (xls)."""
    if Path(ruta).suffix.lower() in (".xlsx", ".xlsm"):
        wb = openpyxl.load_workbook(ruta, data_only=True)  # data_only: resultado de fórmulas, no "=..."
        return [(ws.title, _filas_hoja_openpyxl(ws)) for ws in wb.worksheets]
    xl = pd.ExcelFile(ruta)
    return [(h, xl.parse(h, header=None).values.tolist()) for h in xl.sheet_names]


def leer_excel_detectando_cabecera(ruta, grupos_obligatorios, max_filas=30, todas_las_hojas=False):
    """Busca en cada hoja la fila de cabecera que contiene todos los grupos de columnas.
    - Propaga celdas combinadas y detecta el guion contable.
    - todas_las_hojas=True concatena TODAS las hojas que tengan esa cabecera."""
    tablas = []
    for hoja, filas in _hojas_como_filas(ruta):
        for i, fila in enumerate(filas[:max_filas]):
            valores = [v for v in fila if not es_vacio(v)]
            if valores and all(buscar_columna(valores, g, obligatoria=False) for g in grupos_obligatorios):
                df = pd.DataFrame(filas[i + 1:], columns=_columnas_unicas(fila))
                df = df.dropna(how="all")
                df = df[~df.apply(lambda r: all(es_vacio(v) for v in r), axis=1)]
                df["_hoja"] = hoja
                print(f"   -> '{Path(ruta).name}': hoja '{hoja}', cabecera en fila {i + 1}, {len(df)} filas")
                tablas.append(df)
                break
        if tablas and not todas_las_hojas:
            break
    if not tablas:
        raise ValueError(f"No se encontró una cabecera con {grupos_obligatorios} en {ruta}")
    if len(tablas) == 1:
        return tablas[0]
    columnas_comunes = [c for c in tablas[0].columns if all(c in t.columns for t in tablas)]
    df = pd.concat([t[columnas_comunes] for t in tablas], ignore_index=True)
    antes = len(df)
    df = df.drop_duplicates(subset=[c for c in columnas_comunes if c != "_hoja"])
    if antes != len(df):
        print(f"   -> Se quitaron {antes - len(df)} filas duplicadas entre hojas")
    return df


def seleccionar_excel(titulo):
    try:
        import tkinter as tk
        from tkinter import filedialog
    except ImportError:
        print("[!] tkinter no está disponible.")
        return ""
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    ruta = filedialog.askopenfilename(
        title=titulo, filetypes=[("Excel", "*.xlsx *.xlsm *.xls"), ("Todos", "*.*")]
    )
    root.destroy()
    return ruta or ""


def resolver_ruta(ruta, titulo):
    """Usa la ruta si existe; si está vacía o no existe, abre el explorador."""
    if ruta and Path(ruta).exists():
        return ruta
    if ruta:
        print(f"[!] No existe: {ruta}")
    print(f"Seleccione: {titulo}")
    return seleccionar_excel(titulo)


# =============================================================
# 2. MOTOR TF-IDF + MAPEO SIPSE
# =============================================================
class MotorRecuperacion:
    def __init__(self, stop_words=None, n_max=3):
        self.stop_words = stop_words or set()
        self.n_max = n_max
        self.idf = {}
        self.tf_idf_enciclopedia = []
        self.magnitudes_enciclopedia = []
        self.etiquetas_categorias = []
        self.vocabulario_conocido = set()
        self._vocab_lista = []
        self._cache_correcciones = {}

    @staticmethod
    def normalizar(texto):
        if not texto:
            return ""
        texto = quitar_tildes(texto).lower()
        texto = re.sub(r"[^a-z0-9\s]", " ", texto)
        return re.sub(r"\s+", " ", texto).strip()

    def _corregir_palabra(self, p):
        if p in self.vocabulario_conocido or len(p) <= 2:
            return p
        if p not in self._cache_correcciones:
            m = difflib.get_close_matches(p, self._vocab_lista, n=1, cutoff=0.75)
            self._cache_correcciones[p] = m[0] if m else p
        return self._cache_correcciones[p]

    def corregir_ortografia(self, texto):
        return " ".join(self._corregir_palabra(p) for p in self.normalizar(texto).split())

    def obtener_ngramas_combinados(self, texto):
        palabras = [p for p in texto.split() if p not in self.stop_words]
        ngramas = []
        for n in range(1, min(self.n_max, len(palabras)) + 1):
            ngramas.extend(" ".join(palabras[i:i + n]) for i in range(len(palabras) - n + 1))
        return ngramas

    def cargar_enciclopedia(self, nombres_categorias, documentos):
        self.etiquetas_categorias = list(nombres_categorias)
        docs_norm = [self.normalizar(d) for d in documentos]
        for d in docs_norm:
            self.vocabulario_conocido.update(
                p for p in d.split() if p not in self.stop_words and len(p) > 2
            )
        self._vocab_lista = sorted(self.vocabulario_conocido)

        tokenizados = [self.obtener_ngramas_combinados(d) for d in docs_norm]
        df_terminos = Counter(t for toks in tokenizados for t in set(toks))
        N = len(documentos)
        self.idf = {t: math.log10((1.0 + N) / f) for t, f in df_terminos.items()}

        for toks in tokenizados:
            vec = {t: (1.0 + math.log10(c)) * self.idf[t] for t, c in Counter(toks).items()}
            self.tf_idf_enciclopedia.append(vec)
            self.magnitudes_enciclopedia.append(math.sqrt(sum(v * v for v in vec.values())))

    def clasificar_query(self, texto_query, umbral_similitud=0.18, margen_ambiguo=0.05):
        tokens = self.obtener_ngramas_combinados(self.corregir_ortografia(texto_query))
        if not tokens:
            return "-", 0.0, "Sin palabras válidas"
        vq = {t: (1.0 + math.log10(c)) * self.idf[t] for t, c in Counter(tokens).items() if t in self.idf}
        mag_q = math.sqrt(sum(v * v for v in vq.values()))
        if mag_q == 0:
            return "-", 0.0, "Coseno = 0"

        resultados = []
        for idx, vc in enumerate(self.tf_idf_enciclopedia):
            mag_c = self.magnitudes_enciclopedia[idx]
            punto = sum(v * vc.get(t, 0.0) for t, v in vq.items())  # recorre el vector más corto
            resultados.append((self.etiquetas_categorias[idx], punto / (mag_q * mag_c) if mag_c else 0.0))
        resultados.sort(key=lambda x: x[1], reverse=True)
        mejor_cat, mejor = resultados[0]
        debug = " | ".join(f"{c}: {s:.3f}" for c, s in resultados if s > 0) or "0.0 en todas"
        if mejor < umbral_similitud:
            return "-", mejor, f"Bajo umbral ({debug})"
        if len(resultados) > 1 and (mejor - resultados[1][1]) < margen_ambiguo:
            return mejor_cat, mejor, f"Ambiguo (2º={resultados[1][1]:.3f})"
        return mejor_cat, mejor, f"OK ({debug})"


# Orden: de lo MÁS específico a lo MÁS general (gana el primer patrón que coincide).
MAPEO_SIPSE = [
    # ---------- PROYECTO DE INVERSIÓN ----------
    {"estado": "Actualización de criterios", "etapa": "Proyecto de Inversión", "peso_min": 0.20,
     "variantes": [r"actualizacion de criterios", r"criterios de elegibilidad", r"criterios de viabilidad"]},
    {"estado": "Cocreación de propuestas", "etapa": "Proyecto de Inversión", "peso_min": 0.20,
     "variantes": [r"cocreacion", r"construccion conjunta de propuestas"]},
    {"estado": "Actualización DTS/Ficha EBI", "etapa": "Proyecto de Inversión", "peso_min": 0.20,
     "variantes": [r"actualizacion dts", r"actualizacion ficha ebi", r"documento tecnico de soporte"]},
    # ---------- PRECONTRACTUAL ----------
    # "Asistencia Técnica (Central)" va ANTES que la genérica; antes nunca podía coincidir.
    {"estado": "Asistencia Técnica (Central)", "etapa": "Precontractual", "peso_min": 0.22,
     "variantes": [r"asistencia tecnica \(?nivel central\)?", r"asistencia tecnica central"]},
    {"estado": "Asistencia técnica del sector", "etapa": "Proyecto de Inversión", "peso_min": 0.20,
     "variantes": [r"asistencia tecnica", r"apoyo tecnico del sector"]},
    {"estado": "Estudios Previos y anexos", "etapa": "Precontractual", "peso_min": 0.20,
     "variantes": [r"estudios previos", r"documentos anexos"]},
    {"estado": "Comité de Contratación", "etapa": "Precontractual", "peso_min": 0.22,
     "variantes": [r"comite de contratacion", r"sesion del comite", r"reunion del comite"]},
    {"estado": "Publicación proceso", "etapa": "Precontractual", "peso_min": 0.22,
     "variantes": [r"publicacion proceso contractual", r"publicacion del proceso", r"proceso publicado"]},
    {"estado": "Adjudicación y suscripción", "etapa": "Precontractual", "peso_min": 0.20,
     "variantes": [r"adjudicacion y suscripcion", r"adjudicacion del contrato", r"suscripcion del contrato"]},
    # ---------- POSTCONTRACTUAL (antes que "En ejecución" para que "ejecución finalizada 100%" no caiga ahí) ----------
    {"estado": "Finalización ejecución", "etapa": "Postcontractual", "peso_min": 0.22,
     "variantes": [r"finalizacion ejecucion contrato", r"finalizacion de la ejecucion",
                   r"contrato finalizado", r"ejecucion finalizada"]},
    {"estado": "En proceso de liquidación", "etapa": "Postcontractual", "peso_min": 0.22,
     "variantes": [r"proceso de liquidacion", r"liquidacion del contrato", r"contrato en liquidacion"]},
    # ---------- EJECUCIÓN CONTRACTUAL ----------
    {"estado": "Inicio ejecución contrato", "etapa": "Ejecución Contractual", "peso_min": 0.22,
     "variantes": [r"inicio ejecucion del contrato", r"inicio de ejecucion", r"contrato iniciado",
                   r"se dio inicio al contrato"]},
    {"estado": "En Modificación contractual", "etapa": "Ejecución Contractual", "peso_min": 0.22,
     "variantes": [r"modificacion contractual", r"modificacion al contrato", r"adicion al contrato", r"otrosi"]},
    {"estado": "En ejecución", "etapa": "Ejecución Contractual", "peso_min": 0.22,
     "variantes": [r"en ejecucion", r"en la ejecucion", r"ejecutando el contrato", r"contrato en curso",
                   r"en desarrollo del contrato", r"ejecucion contractual",
                   r"(?:ejecuc|ejecutad|contrato).{0,40}?\d{1,3}\s*(?:%|por\s*ciento)",
                   r"\d{1,3}\s*(?:%|por\s*ciento).{0,40}?(?:ejecuc|ejecutad|avance)"]},
]
for _b in MAPEO_SIPSE:  # precompilar una sola vez
    _b["regex"] = [re.compile(v) for v in _b["variantes"]]

PREFIJO_ETAPA = re.compile(r"^\s*(?:etapa|fase|num|n[°º]|item|paso)?\s*\d{1,2}\s*[\.\)\-:]?\s*", re.IGNORECASE)


def analizar_observacion_estricta(texto_crudo, motor=None):
    """Retorna (cumple, etapa, estado). cumple: 'Sí' | 'Sí (por TF-IDF)' | 'No'."""
    if es_vacio(texto_crudo):
        return "No", "Sin Etapa", "Sin Estado"
    texto = re.sub(r"\s+", " ", quitar_tildes(texto_crudo).lower()).strip()
    texto = PREFIJO_ETAPA.sub("", texto, count=1)

    for bloque in MAPEO_SIPSE:
        if any(rx.search(texto) for rx in bloque["regex"]):
            return "Sí", bloque["etapa"], bloque["estado"]

    if motor is not None:
        categoria, score, _ = motor.clasificar_query(texto_crudo)
        for bloque in MAPEO_SIPSE:
            if bloque["estado"] == categoria and score >= bloque["peso_min"]:
                return "Sí (por TF-IDF)", bloque["etapa"], bloque["estado"]
    return "No", "Sin Etapa", "Sin Estado"


PATRON_ENLACE = re.compile(r"(?i)(https?://|drive\.google|www\.|\.pdf|\.docx?|\.xlsx?)")


def evaluar_evidencia(texto_medio, min_caracteres):
    """'Sí (Detectada)' si hay enlace/archivo o más de `min_caracteres` caracteres."""
    if es_vacio(texto_medio):
        return "No (Falta)"
    texto = str(texto_medio).strip()
    if PATRON_ENLACE.search(texto) or len(texto) > min_caracteres:
        return "Sí (Detectada)"
    return "No (Insuficiente)"


STOP_COHERENCIA = {"para", "como", "este", "esta", "pero", "porque", "cuando", "donde", "quien",
                   "desde", "hacia", "sobre", "entre", "hasta", "segun", "tiene", "todo", "parte",
                   "cual", "anos", "2024", "2025", "2026", "2027"}


def evaluar_coherencia_textual(obs_texto, medio_texto):
    if es_vacio(medio_texto):
        return "Sin texto"
    if es_vacio(obs_texto):
        return "Observación vacía (No evaluable)"
    medio = quitar_tildes(medio_texto).lower()
    if PATRON_ENLACE.search(medio):
        return "Coherente (Enlace o Archivo detectado)"
    obs = quitar_tildes(obs_texto).lower()
    comunes = (set(re.findall(r"\b[a-z0-9]{4,}\b", obs)) & set(re.findall(r"\b[a-z0-9]{4,}\b", medio))) - STOP_COHERENCIA
    if comunes:
        return f"Coherente ({len(comunes)} palabras en común)"
    return "Incoherente (Sin relación textual)"


def cargar_stopwords(ruta):
    try:
        df = pd.read_excel(ruta)
        col = next((c for c in df.columns if str(c).lower() == "stopword"), df.columns[0])
        sw = {str(x).strip().lower() for x in df[col].dropna() if str(x).strip()}
        print(f"Stopwords cargadas: {len(sw)}")
        return sw
    except Exception as e:
        print(f"[!] No se pudieron cargar stopwords: {e}")
        return set()


def cargar_enciclopedia(ruta):
    try:
        df = pd.read_excel(ruta)
        cat = buscar_columna(df.columns, ["Categoria"], obligatoria=False) or df.columns[0]
        txt = (buscar_columna(df.columns, ["Texto", "Ejemplo", "Patron"], obligatoria=False)
               or df.columns[min(1, len(df.columns) - 1)])
        df = df.dropna(subset=[cat, txt])
        print(f"Enciclopedia cargada: {len(df)} documentos")
        return df[cat].astype(str).str.strip().tolist(), df[txt].astype(str).str.strip().tolist()
    except Exception as e:
        print(f"[!] No se pudo cargar enciclopedia: {e}")
        return [], []


STOPWORDS_BASICAS = {"de", "la", "el", "los", "las", "y", "en", "a", "al", "del", "con", "por", "para",
                     "un", "una", "se", "que", "su", "sus", "o", "lo", "es"}


def _enciclopedia_desde_mapeo():
    """Un documento por estado, armado con sus frases de MAPEO_SIPSE (sin los patrones regex)."""
    etiquetas, textos = [], []
    for b in MAPEO_SIPSE:
        limpias = (v.replace("\\(", "").replace("\\)", "").replace("?", "") for v in b["variantes"])
        frases = [v for v in limpias if not re.search(r"[\[\]{}.*+|\\]", v)]
        etiquetas.append(b["estado"])
        textos.append(" ".join(frases))
    return etiquetas, textos


def construir_motor():
    """El motor TF-IDF se construye siempre con MAPEO_SIPSE. La enciclopedia (Excel) es OPCIONAL:
    si existe, sus ejemplos se suman. Debe tener dos columnas:
        Categoria -> nombre EXACTO de un estado de MAPEO_SIPSE (ej. 'Comité de Contratación')
        Texto     -> una observación real de ejemplo escrita por una localidad para ese estado."""
    etiquetas, textos = _enciclopedia_desde_mapeo()
    ruta_enc = CONFIG.get("ruta_enciclopedia")
    if ruta_enc and Path(ruta_enc).exists():
        estados = {b["estado"] for b in MAPEO_SIPSE}
        extra_e, extra_t = cargar_enciclopedia(ruta_enc)
        invalidas = sorted({e for e in extra_e if e not in estados})
        if invalidas:
            print(f"   [!] Categorías de la enciclopedia que no existen en MAPEO_SIPSE (se ignoran): {invalidas}")
        for e, t in zip(extra_e, extra_t):
            if e in estados:
                etiquetas.append(e)
                textos.append(t)
    ruta_sw = CONFIG.get("ruta_stopwords")
    stopwords = cargar_stopwords(ruta_sw) if ruta_sw and Path(ruta_sw).exists() else STOPWORDS_BASICAS
    motor = MotorRecuperacion(stop_words=stopwords, n_max=3)
    motor.cargar_enciclopedia(etiquetas, textos)
    print(f"Motor TF-IDF inicializado con {len(etiquetas)} documentos.")
    return motor


# =============================================================
# 3 y 4. ARCHIVOS .TXT POR AÑO (exclusiones y metas a reportar)
# =============================================================
_CACHE_PLAN, _CACHE_SIPSE, _CACHE_EXCLUIDAS, _CACHE_A_REPORTAR = {}, {}, {}, {}

COLS_TXT_EXCLUIDAS = ["codigo_localidad", "codigo_meta_original", "nombre_meta_original",
                      "codigo_meta_sipse", "nombre_meta_sipse", "proyecto_sipse", "similitud", "nota"]
COLS_TXT_A_REPORTAR = ["año", "codigo_localidad", "metas_a_reportar_mes"]


def ruta_txt(prefijo, año):
    return Path(CONFIG["carpeta_txt"]) / f"{prefijo}{año}.txt"


def guardar_txt(df, ruta):
    df.to_csv(ruta, sep="\t", index=False, encoding="utf-8-sig")
    print(f"   -> Archivo creado: {ruta}")


def leer_txt(ruta):
    print(f"   -> Leyendo archivo existente: {ruta}")
    return pd.read_csv(ruta, sep="\t", dtype=str, encoding="utf-8-sig").fillna("")


def usar_txt_existente(ruta):
    """True si el .txt existe y no se pidió regenerarlo. Si se regenera, el anterior se guarda como .bak."""
    if not ruta.exists():
        return False
    if CONFIG.get("regenerar_txt"):
        respaldo = ruta.with_suffix(".txt.bak")
        ruta.replace(respaldo)
        print(f"   -> regenerar_txt=True: el archivo anterior se guardó como {respaldo.name}")
        return False
    return True


def obtener_plan(año):
    """Excel de planeación de metas del año (se pide una sola vez y se reutiliza para ambos .txt)."""
    if año not in _CACHE_PLAN:
        ruta = resolver_ruta(CONFIG["rutas_plan_metas"].get(año, ""),
                             f"Excel de metas {año} (columna 'Magnitud Meta anualizada DEFINITIVA {año}')")
        if not ruta:
            _CACHE_PLAN[año] = None
        else:
            df = leer_excel_detectando_cabecera(
                ruta, [COLS_PLAN["localidad"], COLS_PLAN["nombre_meta"], col_magnitud(año)],
                todas_las_hojas=LEER_TODAS_LAS_HOJAS_PLAN)
            c_loc = buscar_columna(df.columns, COLS_PLAN["localidad"])
            c_mag = buscar_columna(df.columns, col_magnitud(año))
            c_nom = buscar_columna(df.columns, COLS_PLAN["nombre_meta"])
            total_filas = len(df)
            # Solo filas que realmente son metas (tienen nombre)
            df = df[~df[c_nom].apply(es_vacio)].copy()
            # Celdas combinadas: el código de localidad suele venir solo en la primera fila del grupo
            df["_cod_loc"] = df[c_loc].apply(codigo_localidad).ffill()
            sin_loc = df["_cod_loc"].isna().sum()
            if sin_loc:
                print(f"   [!] {sin_loc} metas sin código de localidad reconocible (se omiten).")
            df = df[df["_cod_loc"].notna()].copy()
            df["_cod_loc"] = df["_cod_loc"].astype(int)
            # Excluida = magnitud vacía, "-" o 0 (el formato contable muestra "-" pero el valor es 0)
            df["_motivo_excl"] = df[c_mag].apply(motivo_exclusion_magnitud)
            df["_excluida"] = df["_motivo_excl"].notna()
            print(f"   -> Columna de magnitud usada: '{c_mag}'")
            print(f"   -> Plan {año}: {total_filas} filas leídas, {len(df)} metas, "
                  f"{int(df['_excluida'].sum())} excluidas, {int((~df['_excluida']).sum())} a reportar")
            _CACHE_PLAN[año] = df
    return _CACHE_PLAN[año]


def obtener_consulta_sipse(año):
    if año not in _CACHE_SIPSE:
        ruta = resolver_ruta(CONFIG["rutas_consulta_sipse"].get(año, ""),
                             f"Excel 'Consulta avance de metas' SIPSE {año}")
        if not ruta:
            _CACHE_SIPSE[año] = None
        else:
            df = leer_excel_detectando_cabecera(
                ruta, [COLS_SIPSE["localidad"], COLS_SIPSE["cod_meta"], COLS_SIPSE["nombre_meta"]])
            c_vig = buscar_columna(df.columns, COLS_SIPSE["vigencia"], obligatoria=False)
            if c_vig:
                df = df[df[c_vig].apply(a_entero) == año]
            _CACHE_SIPSE[año] = df
    return _CACHE_SIPSE[año]


PALABRAS_VACIAS_META = {"de", "la", "el", "los", "las", "y", "e", "en", "a", "al", "del", "con", "por",
                        "para", "un", "una", "unos", "unas", "o", "u", "que", "se", "su", "sus"}


class IndiceTfidfMetas:
    """TF-IDF sobre palabras + trigramas de caracteres. Los trigramas toleran errores de digitación,
    plurales y abreviaturas ('comunales' vs 'comunal'), y las palabras pesan los términos distintivos."""

    def __init__(self, textos):
        docs = [self._rasgos(t) for t in textos]
        n = len(docs)
        df_ = Counter(r for d in docs for r in set(d))
        self.idf = {r: math.log((1 + n) / (1 + f)) + 1.0 for r, f in df_.items()}
        self.vectores = [self._vector(d) for d in docs]

    @staticmethod
    def _rasgos(texto):
        t = re.sub(r"[^a-z0-9]+", " ", quitar_tildes(texto).lower())
        palabras = [w for w in t.split() if w not in PALABRAS_VACIAS_META]
        rasgos = [f"w:{w}" for w in palabras]
        for w in palabras:
            w = f" {w} "
            rasgos.extend(f"c:{w[i:i + 3]}" for i in range(len(w) - 2))
        return rasgos

    def _vector(self, rasgos):
        vec = {r: (1.0 + math.log(c)) * self.idf.get(r, 1.0) for r, c in Counter(rasgos).items()}
        mag = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {r: v / mag for r, v in vec.items()}

    def puntajes(self, texto):
        q = self._vector(self._rasgos(texto))
        return [sum(v * d.get(r, 0.0) for r, v in q.items()) for d in self.vectores]


def emparejar_uno_a_uno(textos_plan, textos_sipse):
    """Asigna a CADA meta del plan una meta de SIPSE.
    1) Asignación uno a uno por mayor similitud (una meta SIPSE no se reparte entre dos del plan).
    2) Si hay más metas en el plan que en SIPSE, las sobrantes reciben su mejor opción (marcada).
    Devuelve lista de (indice_sipse, similitud, nota)."""
    if not textos_sipse:
        return [(None, 0.0, "Localidad sin metas en SIPSE")] * len(textos_plan)
    indice = IndiceTfidfMetas(textos_sipse)
    matriz = [indice.puntajes(t) for t in textos_plan]
    pares = sorted(((s, i, j) for i, fila in enumerate(matriz) for j, s in enumerate(fila)), reverse=True)
    asignado, usados = {}, set()
    for s, i, j in pares:
        if i not in asignado and j not in usados:
            asignado[i] = (j, s, "")
            usados.add(j)
    for i, fila in enumerate(matriz):
        if i not in asignado:
            j = max(range(len(fila)), key=fila.__getitem__)
            asignado[i] = (j, fila[j], "Meta SIPSE compartida (hay más metas en el plan que en SIPSE)")
    return [asignado[i] for i in range(len(textos_plan))]


def construir_txt_excluidas(año, destino):
    print(f"Creando {destino.name} ...")
    plan = obtener_plan(año)
    sipse = obtener_consulta_sipse(año) if plan is not None else None
    if plan is None or sipse is None:
        print(f"[!] No se seleccionaron los archivos fuente; no se crea {destino.name}.")
        return None

    c_nom = buscar_columna(plan.columns, COLS_PLAN["nombre_meta"])
    c_cod = buscar_columna(plan.columns, COLS_PLAN["cod_meta"], obligatoria=False)
    if not c_cod:
        print("[!] No se halló la columna de código de meta en el plan; 'codigo_meta_original' quedará vacío.")

    s_loc = buscar_columna(sipse.columns, COLS_SIPSE["localidad"])
    s_cod = buscar_columna(sipse.columns, COLS_SIPSE["cod_meta"])
    s_nom = buscar_columna(sipse.columns, COLS_SIPSE["nombre_meta"])
    s_proy = buscar_columna(sipse.columns, COLS_SIPSE["proyecto"], obligatoria=False)
    print(f"   -> SIPSE: localidad='{s_loc}', código='{s_cod}', nombre='{s_nom}', proyecto='{s_proy}'")

    # Metas únicas de SIPSE por localidad. Se deduplica por localidad + proyecto + código + NOMBRE:
    # el mismo número de meta puede repetirse en proyectos distintos y no debe perderse ninguna.
    sipse = sipse.assign(_loc=sipse[s_loc].apply(codigo_localidad),
                         _cod=sipse[s_cod].apply(normalizar_codigo),
                         _proy=sipse[s_proy].apply(normalizar_codigo) if s_proy else "",
                         _nom=sipse[s_nom].astype(str).str.strip())
    sin_loc = sipse["_loc"].isna().sum()
    if sin_loc:
        print(f"   [!] {sin_loc} filas de SIPSE sin localidad reconocible "
              f"(ej.: {sipse.loc[sipse['_loc'].isna(), s_loc].astype(str).unique()[:3].tolist()})")
    sipse = sipse[sipse["_loc"].notna() & ~sipse[s_nom].apply(es_vacio)]
    sipse = sipse.drop_duplicates(subset=["_loc", "_proy", "_cod", "_nom"])
    print(f"   -> SIPSE {año}: {len(sipse)} metas únicas para emparejar")

    filas = []
    for loc, grupo_plan in plan.groupby("_cod_loc"):
        grupo_sipse = sipse[sipse["_loc"] == loc]
        cods_s = grupo_sipse["_cod"].tolist()
        noms_s = grupo_sipse["_nom"].tolist()
        proys_s = grupo_sipse["_proy"].tolist()
        # Se emparejan TODAS las metas del plan (incluidas y excluidas) para que una meta excluida
        # no "robe" el código de una meta vigente; luego se conservan solo las excluidas.
        noms_p = grupo_plan[c_nom].astype(str).str.strip().tolist()
        asignaciones = emparejar_uno_a_uno(noms_p, noms_s)
        for (_, r), (j, s, nota) in zip(grupo_plan.iterrows(), asignaciones):
            if not r["_excluida"]:
                continue
            if s < UMBRAL_AVISO_SIMILITUD and not nota:
                nota = "Similitud baja: revisar"
            filas.append([f"{loc:02d}", normalizar_codigo(r[c_cod]) if c_cod else "", str(r[c_nom]).strip(),
                          cods_s[j] if j is not None else "", noms_s[j] if j is not None else "",
                          proys_s[j] if j is not None else "", f"{s:.2f}", nota])

    df = pd.DataFrame(filas, columns=COLS_TXT_EXCLUIDAS)
    guardar_txt(df, destino)
    n_plan = int(plan["_excluida"].sum())
    n_revisar = (df["nota"] != "").sum()
    print(f"   -> {len(df)} de {n_plan} metas excluidas del plan guardadas ({n_revisar} con nota para revisar).")
    return df


def construir_txt_a_reportar(año, destino):
    print(f"Creando {destino.name} ...")
    plan = obtener_plan(año)
    if plan is None:
        print(f"[!] No se seleccionó el archivo de metas; no se crea {destino.name}.")
        return None
    conteo = plan[~plan["_excluida"]].groupby("_cod_loc").size().reset_index(name="metas_a_reportar_mes")
    df = pd.DataFrame({"año": año,
                       "codigo_localidad": conteo["_cod_loc"].map(lambda x: f"{x:02d}"),
                       "metas_a_reportar_mes": conteo["metas_a_reportar_mes"]})
    guardar_txt(df, destino)
    return df.astype(str)


def obtener_metas_excluidas(año):
    """Devuelve dos conjuntos para la vigencia `año`:
    - con_proyecto: {(cod_localidad, proyecto, codigo_meta_sipse)} cuando el .txt trae proyecto.
    - sin_proyecto: {(cod_localidad, codigo_meta_sipse)} para filas del .txt sin proyecto."""
    if año in _CACHE_EXCLUIDAS:
        return _CACHE_EXCLUIDAS[año]
    con_proyecto, sin_proyecto = set(), set()
    if año is not None and año >= AÑO_INICIO_PLAN:
        ruta = ruta_txt("metasExcluidas", año)
        df = leer_txt(ruta) if usar_txt_existente(ruta) else construir_txt_excluidas(año, ruta)
        if df is not None:
            df = df.astype(str)
            if "proyecto_sipse" not in df.columns:
                df["proyecto_sipse"] = ""
            sin_codigo = 0
            for l, p, m in zip(df["codigo_localidad"], df["proyecto_sipse"], df["codigo_meta_sipse"]):
                if not m.strip():
                    sin_codigo += 1
                    continue
                loc, m = codigo_localidad(l), normalizar_codigo(m)
                if p.strip():
                    con_proyecto.add((loc, normalizar_codigo(p), m))
                else:
                    sin_proyecto.add((loc, m))
            print(f"Metas excluidas {año}: {len(df)} filas en {ruta.name} "
                  f"({len(con_proyecto) + len(sin_proyecto)} códigos SIPSE distintos)")
            if sin_codigo:
                print(f"   [!] {sin_codigo} metas excluidas sin código SIPSE en el .txt "
                      f"(su localidad no tiene metas en la consulta SIPSE). Complételas a mano.")
    _CACHE_EXCLUIDAS[año] = (con_proyecto, sin_proyecto)
    return _CACHE_EXCLUIDAS[año]


def es_meta_excluida(año, cod_loc, proyecto, meta):
    con_proyecto, sin_proyecto = obtener_metas_excluidas(año)
    return (cod_loc, proyecto, meta) in con_proyecto or (cod_loc, meta) in sin_proyecto


def obtener_metas_a_reportar(año):
    """{cod_localidad: cantidad} o None si ese año no tiene archivo (se usa lo reportado como denominador)."""
    if año in _CACHE_A_REPORTAR:
        return _CACHE_A_REPORTAR[año]
    resultado = None
    if año is not None and año >= AÑO_INICIO_PLAN:
        ruta = ruta_txt("metasAReportar", año)
        df = leer_txt(ruta) if usar_txt_existente(ruta) else construir_txt_a_reportar(año, ruta)
        if df is not None:
            resultado = {codigo_localidad(l): int(float(c))
                         for l, c in zip(df["codigo_localidad"], df["metas_a_reportar_mes"])}
    _CACHE_A_REPORTAR[año] = resultado
    return resultado


# =============================================================
# 5. REGLAS DE CALIFICACIÓN POR AÑO
# =============================================================
@dataclass
class ResultadoEvaluacion:
    regla: str
    cumple_estructura: str
    etapa: str
    estado: str
    evidencia: str
    coherencia: str
    cumple_final: bool


def regla_hasta_2025(obs, medio, motor):
    """Hasta 2025: SIN restricción de estructura ni de longitud.
    Cumple si OBSERVACIONES y MEDIO DE VERIFICACIÓN tienen cualquier texto escrito."""
    tiene_obs, tiene_medio = not es_vacio(obs), not es_vacio(medio)
    return ResultadoEvaluacion(
        regla="Hasta 2025: obs. y evidencia diligenciadas",
        cumple_estructura="Sí" if tiene_obs else "No",
        etapa="N/A", estado="N/A",
        evidencia="Sí (Detectada)" if tiene_medio else "No (Falta)",
        coherencia=evaluar_coherencia_textual(obs, medio),
        cumple_final=tiene_obs and tiene_medio,
    )


def regla_2026(obs, medio, motor):
    """2026: observación con estructura SIPSE (etapa/estado) y evidencia con más de 10 caracteres."""
    cumple, etapa, estado = analizar_observacion_estricta(obs, motor)
    evidencia = evaluar_evidencia(medio, min_caracteres=10)
    return ResultadoEvaluacion(
        regla="2026: estructura + evidencia >10 caracteres",
        cumple_estructura=cumple, etapa=etapa, estado=estado, evidencia=evidencia,
        coherencia=evaluar_coherencia_textual(obs, medio),
        cumple_final=cumple.startswith("Sí") and evidencia == "Sí (Detectada)",
    )


# Para agregar una nueva vigencia: escriba `def regla_2027(obs, medio, motor): ...`
# y regístrela aquí. Cada clave aplica desde ese año hasta la siguiente clave.
REGLAS_POR_AÑO = {
    2025: regla_hasta_2025,   # también se usa para años anteriores
    2026: regla_2026,
    # 2027: regla_2027,
}
_AVISOS_REGLA = set()


def obtener_regla(año):
    años = sorted(REGLAS_POR_AÑO)
    if año is None:
        return REGLAS_POR_AÑO[años[-1]]
    aplicables = [a for a in años if a <= año]
    clave = aplicables[-1] if aplicables else años[0]
    if año > años[-1] and año not in _AVISOS_REGLA:
        print(f"[!] No hay regla definida para {año}; se aplica la de {clave}.")
        _AVISOS_REGLA.add(año)
    return REGLAS_POR_AÑO[clave]


# =============================================================
# 6. PROCESO PRINCIPAL
# =============================================================
def _norm_cabecera(valor):
    return re.sub(r"\s+", " ", quitar_tildes(str(valor or "")).upper()).strip()


def localizar_columnas(ws, max_filas=15):
    """Devuelve (fila_cabecera, col_obs, col_medio, {nombre: col}) buscando en una sola fila."""
    for r in range(1, max_filas + 1):
        fila = {c: _norm_cabecera(ws.cell(row=r, column=c).value) for c in range(1, ws.max_column + 1)}
        col_obs = next((c for c, v in fila.items() if v == "OBSERVACIONES"), None)
        if not col_obs:
            continue
        col_medio = next((c for c, v in fila.items() if "MEDIO" in v and "VERIFICA" in v), None)
        indices = {n: c for c, v in fila.items() for n in COLS_REQUERIDAS_DATOS if v == n}
        return r, col_obs, col_medio, indices
    return None, None, None, {}


def procesar_sabana_observaciones():
    ruta_excel = resolver_ruta(CONFIG["ruta_datos"], "Sábana de observaciones SIPSE")
    if not ruta_excel:
        print("Proceso cancelado: no se seleccionó la sábana.")
        return
    print("Leyendo Excel (modo seguro para conservar logos y diseño)...")
    wb = openpyxl.load_workbook(ruta_excel)
    ws = wb.active

    fila_cab, col_obs, col_medio, idx = localizar_columnas(ws)
    if not fila_cab:
        print("Error: no se encontró la columna OBSERVACIONES.")
        return
    faltantes = [n for n in ("ENTIDAD", "VIGENCIA", "MES", "META") if n not in idx]
    if faltantes:
        print(f"Error: faltan columnas indispensables: {faltantes}")
        return
    if not col_medio:
        print("[!] No se encontró la columna de MEDIO DE VERIFICACIÓN.")
    print(f"Cabeceras en fila {fila_cab}.")

    motor = construir_motor()

    # Insertar columnas nuevas a la derecha de OBSERVACIONES
    n_nuevas = len(NUEVAS_COLUMNAS)
    ws.insert_cols(col_obs + 1, amount=n_nuevas)
    for i, nombre in enumerate(NUEVAS_COLUMNAS, start=1):
        ws.cell(row=fila_cab, column=col_obs + i, value=nombre)
    if col_medio and col_medio > col_obs:
        col_medio += n_nuevas
    idx = {n: (c + n_nuevas if c > col_obs else c) for n, c in idx.items()}

    print("Excluyendo metas y evaluando observaciones...")
    registros = []
    for row in range(fila_cab + 1, ws.max_row + 1):
        valores = {n: ws.cell(row=row, column=c).value for n, c in idx.items()}

        if any(es_vacio(v) for v in valores.values()):
            for i in range(1, n_nuevas + 1):
                ws.cell(row=row, column=col_obs + i, value="-")
            continue

        año = a_entero(valores["VIGENCIA"])
        cod_loc = codigo_localidad(limpiar_nombre_localidad(str(valores["ENTIDAD"])))
        meta = normalizar_codigo(valores["META"])
        registro = {"COD_LOC": cod_loc, "VIGENCIA": año, "MES": valores["MES"], "META": meta,
                    "EXCLUIDA": False, "CUMPLE": False}

        # 1) Exclusión ANTES de evaluar la estructura
        proyecto = normalizar_codigo(valores.get("NUMPROYECTO"))
        if cod_loc is not None and es_meta_excluida(año, cod_loc, proyecto, meta):
            registro["EXCLUIDA"] = True
            salida = ["Meta excluida"] * n_nuevas
        else:
            obs = ws.cell(row=row, column=col_obs).value
            medio = ws.cell(row=row, column=col_medio).value if col_medio else None
            res = obtener_regla(año)(obs, medio, motor)
            registro["CUMPLE"] = res.cumple_final
            coherencia = res.coherencia if col_medio else "Columna Medio Verif no hallada"
            salida = [res.cumple_estructura, res.etapa, res.estado, res.evidencia, coherencia]

        for i, v in enumerate(salida, start=1):
            ws.cell(row=row, column=col_obs + i, value=v)
        registros.append(registro)

    ruta = Path(ruta_excel)
    nueva_ruta = ruta.with_name(f"{ruta.stem}{SUFIJO_SALIDA}")
    wb.save(nueva_ruta)
    print("Evaluación fila por fila terminada. Generando reportes...")

    df_resumen, df_matriz = calcular_resumen(pd.DataFrame(registros))
    escribir_reportes(nueva_ruta, df_resumen, df_matriz)
    print(f"¡Análisis completado! Archivo guardado en:\n{nueva_ruta}")


def calcular_resumen(df):
    claves = ["COD_LOC", "VIGENCIA", "MES"]
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()
    # Solo localidades 01-20 (Nivel Central y no reconocidas quedan fuera)
    df = df[df["COD_LOC"].notna()].copy()
    df["COD_LOC"] = df["COD_LOC"].astype(int)

    excluidas = df[df["EXCLUIDA"]].groupby(claves)["META"].nunique().rename("Metas_Excluidas")
    df = df[~df["EXCLUIDA"]]
    reportadas = df.groupby(claves)["META"].nunique().rename("Metas_Reportadas")
    cumplen = df[df["CUMPLE"]].groupby(claves)["META"].nunique().rename("Metas_Cumplen")

    # Denominador: metas que DEBEN reportarse (archivo .txt). Se arma la malla localidad x mes
    # para que una localidad que no reportó nada aparezca con 0 %.
    bases = []
    for año in sorted(a for a in df["VIGENCIA"].dropna().unique()):
        meses = df.loc[df["VIGENCIA"] == año, "MES"].unique()
        esperadas = obtener_metas_a_reportar(int(año))
        if esperadas:
            bases.append(pd.DataFrame(
                [(loc, año, mes, n, "Archivo metasAReportar") for loc, n in esperadas.items() if loc
                 for mes in meses],
                columns=claves + ["Metas_Esperadas", "Fuente_Denominador"]))
        else:
            r = reportadas.reset_index()
            r = r[r["VIGENCIA"] == año].rename(columns={"Metas_Reportadas": "Metas_Esperadas"})
            r["Fuente_Denominador"] = "Metas reportadas (sin archivo de metas a reportar)"
            bases.append(r)
    res = pd.concat(bases, ignore_index=True)
    for serie in (reportadas, cumplen, excluidas):
        res = res.merge(serie.reset_index(), on=claves, how="left")
    for c in ("Metas_Reportadas", "Metas_Cumplen", "Metas_Excluidas"):
        res[c] = res[c].fillna(0).astype(int)

    res["% Avance Meta 8"] = (res["Metas_Cumplen"] / res["Metas_Esperadas"].where(res["Metas_Esperadas"] > 0)
                              * 100).clip(upper=100).round(2)
    res["ENTIDAD"] = res["COD_LOC"].map(ETIQUETA_POR_CODIGO)
    res["VIGENCIA"] = res["VIGENCIA"].astype(int)
    res["_orden_mes"] = res["MES"].map(orden_mes)
    res = res.sort_values(["VIGENCIA", "_orden_mes", "COD_LOC"])

    # Matriz con columnas (VIGENCIA, MES) como el formato original, meses en orden cronológico
    orden_cols = list(dict.fromkeys(zip(res["VIGENCIA"], res["MES"])))
    matriz = (res.pivot_table(index="ENTIDAD", columns=["VIGENCIA", "MES"], values="% Avance Meta 8")
              .reindex(index=[ETIQUETA_POR_CODIGO[i] for i in range(1, 21)],
                       columns=pd.MultiIndex.from_tuples(orden_cols, names=["VIGENCIA", "MES"])))

    # Hoja de resumen con los rótulos ORIGINALES.
    # Total_Metas_Diferentes = metas que la localidad DEBE reportar (archivo metasAReportar).
    res = res.rename(columns={"Metas_Esperadas": "Total_Metas_Diferentes"})
    res = res[["ENTIDAD", "VIGENCIA", "MES", "Total_Metas_Diferentes", "Metas_Cumplen", "% Avance Meta 8"]]
    return res, matriz


def escribir_reportes(ruta, df_resumen, df_matriz):
    """Mismo formato del código original."""
    with pd.ExcelWriter(ruta, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        df_resumen.to_excel(writer, sheet_name="Resumen Meta 8", index=False)
        if df_matriz.empty:
            return
        df_matriz.to_excel(writer, sheet_name="Matriz Localidades", startcol=1)

        ws_matriz = writer.sheets["Matriz Localidades"]
        ws_matriz.delete_rows(3)

        max_fila = ws_matriz.max_row
        ws_matriz.merge_cells(start_row=3, start_column=1, end_row=max_fila, end_column=1)

        celda_entidad = ws_matriz.cell(row=3, column=1, value="Entidad")
        celda_entidad.font = Font(bold=True, size=12)
        celda_entidad.alignment = Alignment(textRotation=90, horizontal="center", vertical="center")

        max_length = max((len(str(c.value)) for c in ws_matriz["B"] if c.value), default=0)
        ws_matriz.column_dimensions["B"].width = max(max_length + 3, 12)


if __name__ == "__main__":
    procesar_sabana_observaciones()
#5m 30s sin archivos
#3m 47s con archivos


Leyendo Excel (modo seguro para conservar logos y diseño)...
Cabeceras en fila 1.
Motor TF-IDF inicializado con 14 documentos.
Excluyendo metas y evaluando observaciones...
   -> Leyendo archivo existente: C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\metasExcluidas2026.txt
Metas excluidas 2026: 56 filas en metasExcluidas2026.txt (54 códigos SIPSE distintos)
Evaluación fila por fila terminada. Generando reportes...
   -> Leyendo archivo existente: C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\metasAReportar2026.txt
¡Análisis completado! Archivo guardado en:
C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\Salida_1_AnalizadoV1_4.xlsx


### Con conjunto de datos Ing. Luis Alejandro

In [4]:
import os
import re
import shutil
import unicodedata
import openpyxl
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
import pandas as pd

def analizar_observacion_estricta(texto_crudo):
    texto = str(texto_crudo).strip() if texto_crudo else ""
    
    # 1. DICCIONARIO DE ESTADOS Y ETAPAS
    mapeo_sipse = {
        r"actualizaci[oó]n de criterios de elegibilidad y viabilidad": ("Actualización de criterios", "Proyecto de Inversión"),
        r"asistencia t[eé]cnica del sector": ("Asistencia técnica del sector", "Proyecto de Inversión"),
        r"cocreaci[oó]n de propuestas priorizadas.*": ("Cocreación de propuestas", "Proyecto de Inversión"),
        r"actualizaci[oó]n documento t[eé]cnico de soporte.*": ("Actualización DTS/Ficha EBI", "Proyecto de Inversión"),
        
        r"elaboraci[oó]n de estudios previos y documentos anexos": ("Estudios Previos y anexos", "Precontractual"),
        r"comit[eé] de contrataci[oó]n": ("Comité de Contratación", "Precontractual"),
        r"asistencia t[eé]cnica \(nivel central\)": ("Asistencia Técnica (Central)", "Precontractual"),
        r"publicaci[oó]n proceso contractual": ("Publicación proceso", "Precontractual"),
        r"adjudicaci[oó]n y suscripci[oó]n de proceso contractual": ("Adjudicación y suscripción", "Precontractual"),
        
        r"inicio ejecuci[oó]n del contrato": ("Inicio ejecución contrato", "Ejecución Contractual"),
        r"en ejecuci[oó]n \(\d{1,3}%\s*avance\)": ("En ejecución", "Ejecución Contractual"),
        r"en modificaci[oó]n contractual": ("En Modificación contractual", "Ejecución Contractual"),
        
        r"finalizaci[oó]n ejecuci[oó]n contrato": ("Finalización ejecución", "Postcontractual"),
        r"en proceso de liquidaci[oó]n": ("En proceso de liquidación", "Postcontractual")
    }
    
    cumple_estructura = "No"
    estado_extraido = "Sin Estado"
    etapa_extraida = "Sin Etapa"
    
    # 2. EVALUAR ESTRUCTURA Y EXTRAER
    for patron, (nombre_estado, nombre_etapa) in mapeo_sipse.items():
        if re.search(r"(?i)^(?:\d{1,2}\.?\s*-?\s*)?(" + patron + r")", texto):
            cumple_estructura = "Sí"
            estado_extraido = nombre_estado
            etapa_extraida = nombre_etapa
            break 
            
    # 3. EXTRAER EVIDENCIA CONDICIONAL
    patron_evidencia = r"(?i)(http[s]?://|drive\.google|anexo|radicado|pdf|acta|soporte)"
    if re.search(patron_evidencia, texto):
        tiene_evidencia = "Sí (Detectada)"
    else:
        tiene_evidencia = "No (Falta)"
            
    return cumple_estructura, etapa_extraida, estado_extraido, tiene_evidencia


def limpiar_nombre_localidad(entidad):
    """Estandariza el nombre o código numérico de la localidad."""
    if entidad is None or pd.isna(entidad):
        return ""
    
    # Convertir a texto y remover decimales si vienen de Excel (ej: "10.0" -> "10")
    entidad_str = str(entidad).strip().upper()
    entidad_str = re.sub(r'\.0$', '', entidad_str)
    
    nfkd = unicodedata.normalize('NFKD', entidad_str)
    entidad_limpia = "".join([c for c in nfkd if not unicodedata.combining(c)])
    
    if "USAQUEN" in entidad_limpia or entidad_limpia in ["1", "01"]: return "01 - USAQUÉN"
    if "CHAPINERO" in entidad_limpia or entidad_limpia in ["2", "02"]: return "02 - CHAPINERO"
    if "SANTA FE" in entidad_limpia or entidad_limpia in ["3", "03"]: return "03 - SANTA FE"
    if "SAN CRISTOBAL" in entidad_limpia or entidad_limpia in ["4", "04"]: return "04 - SAN CRISTÓBAL"
    if "USME" in entidad_limpia or entidad_limpia in ["5", "05"]: return "05 - USME"
    if "TUNJUELITO" in entidad_limpia or entidad_limpia in ["6", "06"]: return "06 - TUNJUELITO"
    if "BOSA" in entidad_limpia or entidad_limpia in ["7", "07"]: return "07 - BOSA"
    if "KENNEDY" in entidad_limpia or entidad_limpia in ["8", "08"]: return "08 - KENNEDY"
    if "FONTIBON" in entidad_limpia or entidad_limpia in ["9", "09"]: return "09 - FONTIBÓN"
    if "ENGATIVA" in entidad_limpia or entidad_limpia == "10": return "10 - ENGATIVÁ"
    if "SUBA" in entidad_limpia or entidad_limpia == "11": return "11 - SUBA"
    if "BARRIOS UNIDOS" in entidad_limpia or entidad_limpia == "12": return "12 - BARRIOS UNIDOS"
    if "TEUSAQUILLO" in entidad_limpia or entidad_limpia == "13": return "13 - TEUSAQUILLO"
    if "MARTIRES" in entidad_limpia or entidad_limpia == "14": return "14 - LOS MÁRTIRES"
    if "ANTONIO NARINO" in entidad_limpia or entidad_limpia == "15": return "15 - ANTONIO NARIÑO"
    if "PUENTE ARANDA" in entidad_limpia or entidad_limpia == "16": return "16 - PUENTE ARANDA"
    if "CANDELARIA" in entidad_limpia or entidad_limpia == "17": return "17 - LA CANDELARIA"
    if "RAFAEL URIBE" in entidad_limpia or entidad_limpia == "18": return "18 - RAFAEL URIBE URIBE"
    if "CIUDAD BOLIVAR" in entidad_limpia or entidad_limpia == "19": return "19 - CIUDAD BOLÍVAR"
    if "SUMAPAZ" in entidad_limpia or entidad_limpia == "20": return "20 - SUMAPAZ"
    
    return entidad_str


def procesar_sabana_observaciones(ruta_excel, ruta_exclusiones=None):
    nombre_base, ext = os.path.splitext(ruta_excel)
    nueva_ruta = f"{nombre_base}_Analizado.xlsx"
    
    archivo_temporal_creado = False
    ruta_excel_trabajo = ruta_excel

    # 0. ADAPTACIÓN INTELIGENTE DE ARCHIVOS
    if ext.lower() == '.xls':
        print("Detectado archivo con extensión .xls. Verificando formato interno...")
        ruta_excel_trabajo = f"{nombre_base}_temp_convertido.xlsx"
        
        # 99% de las veces en sistemas de gobierno es un .xlsx con la extensión cambiada
        shutil.copyfile(ruta_excel, ruta_excel_trabajo)
        archivo_temporal_creado = True
        
        try:
            wb = openpyxl.load_workbook(ruta_excel_trabajo)
            print("El archivo era internamente un .xlsx. Se preservará el formato original.")
        except Exception:
            # Si falla, entonces sí era un .xls binario muy viejo
            print("Es un archivo .xls antiguo. Convirtiendo datos (se perderá el color/formato original)...")
            df_raw = pd.read_excel(ruta_excel, header=None, engine='xlrd')
            df_raw.to_excel(ruta_excel_trabajo, index=False, header=False, engine='openpyxl')
            wb = openpyxl.load_workbook(ruta_excel_trabajo)
    else:
        print("Leyendo archivo .xlsx...")
        wb = openpyxl.load_workbook(ruta_excel_trabajo)

    ws = wb.active
    
    nombres_requeridos = ["ENTIDAD", "VIGENCIA", "CODIGO", "NOMBRE_META", "Mes Programado"]
    indices_requeridos = {}
    fila_cabecera = None
    col_obs = None
    
    # 1. Buscar dinámicamente las columnas
    for r in range(1, 16):
        for c in range(1, ws.max_column + 1):
            valor_celda = str(ws.cell(row=r, column=c).value).strip()
            if valor_celda.upper() == "OBSERVACIONES":
                fila_cabecera = r
                col_obs = c
            elif valor_celda in nombres_requeridos:
                indices_requeridos[valor_celda] = c
        if fila_cabecera and len(indices_requeridos) == len(nombres_requeridos): 
            break
            
    if not fila_cabecera:
        print("Error: No se encontró la columna OBSERVACIONES.")
        if archivo_temporal_creado and os.path.exists(ruta_excel_trabajo):
            os.remove(ruta_excel_trabajo)
        return

    print(f"Cabeceras encontradas en fila {fila_cabecera}. Preparando estructura...")
    
    # 2. Insertar las 4 columnas de evaluación
    ws.insert_cols(col_obs + 1, amount=4)
    ws.cell(row=fila_cabecera, column=col_obs + 1, value="Cumple_Estructura")
    ws.cell(row=fila_cabecera, column=col_obs + 2, value="Etapa_Extraida")
    ws.cell(row=fila_cabecera, column=col_obs + 3, value="Estado_Extraido")
    ws.cell(row=fila_cabecera, column=col_obs + 4, value="Evidencia_Validada")
    
    print("Analizando observaciones fila por fila...")
    
    # 3. Iterar y procesar filas
    for row in range(fila_cabecera + 1, ws.max_row + 1):
        fila_incompleta = False
        for nombre_col, col_idx in indices_requeridos.items():
            valor = ws.cell(row=row, column=col_idx).value
            if valor is None or str(valor).strip() == "" or str(valor).strip().lower() == "nan":
                fila_incompleta = True
                break
        
        if fila_incompleta:
            ws.cell(row=row, column=col_obs + 1, value="-")
            ws.cell(row=row, column=col_obs + 2, value="-")
            ws.cell(row=row, column=col_obs + 3, value="-")
            ws.cell(row=row, column=col_obs + 4, value="-")
            continue
            
        texto_crudo = ws.cell(row=row, column=col_obs).value
        cumple, etapa, estado, evidencia = analizar_observacion_estricta(texto_crudo)
        
        ws.cell(row=row, column=col_obs + 1, value=cumple)
        ws.cell(row=row, column=col_obs + 2, value=etapa)
        ws.cell(row=row, column=col_obs + 3, value=estado)
        ws.cell(row=row, column=col_obs + 4, value=evidencia)
        
    # 4. Guardar archivo analizado sin dañar el original
    wb.save(nueva_ruta)
    wb.close()
    
    if archivo_temporal_creado and os.path.exists(ruta_excel_trabajo):
        os.remove(ruta_excel_trabajo)
        
    print(f"Evaluación terminada. Guardada en '{nueva_ruta}'. Generando reporte estadístico...")

    # ==========================================
    # 5. CÁLCULO ESTADÍSTICO CON EXCLUSIONES
    # ==========================================
    df = pd.read_excel(nueva_ruta, header=fila_cabecera - 1)
    df_validos = df[df['Cumple_Estructura'] != '-'].copy()

    # Forzar conversión a string antes de hacer el filtro, previene el AttributeError
    df_validos['ENTIDAD'] = df_validos['ENTIDAD'].fillna('').astype(str)

    # Omitir Secretaría de Gobierno Central
    df_validos = df_validos[df_validos['ENTIDAD'].str.strip().str.upper() != 'SECRETARIA DE GOBIERNO - NIVEL CENTRAL']
    
    # Normalizar Localidades
    df_validos['ENTIDAD'] = df_validos['ENTIDAD'].apply(limpiar_nombre_localidad)

    # -------------------------------------------------------------
    # EXCLUSIÓN DINÁMICA DE METAS (ANTI-JOIN)
    # -------------------------------------------------------------
    if ruta_exclusiones:
        try:
            df_excl = pd.read_excel(ruta_exclusiones)
            if 'Entidad' in df_excl.columns:
                df_excl['Entidad'] = df_excl['Entidad'].apply(limpiar_nombre_localidad)
                
            df_excl = df_excl.rename(columns={
                'Año_excluido': 'VIGENCIA',
                'Entidad': 'ENTIDAD',
                'cod_proyecto': 'CODIGO',
                'cod_meta': 'NOMBRE_META'
            })
            
            columnas_cruce = ['VIGENCIA', 'ENTIDAD', 'CODIGO', 'NOMBRE_META']
            
            if all(col in df_excl.columns for col in columnas_cruce):
                for col in columnas_cruce:
                    df_validos[col] = df_validos[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
                    df_excl[col] = df_excl[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
                
                df_validos = df_validos.merge(df_excl[columnas_cruce], on=columnas_cruce, how='left', indicator=True)
                df_validos = df_validos[df_validos['_merge'] == 'left_only'].drop(columns=['_merge'])
                print("Exclusiones aplicadas con éxito.")
            else:
                print("Advertencia: El archivo de exclusiones no contiene las columnas correctas.")
        except Exception as e:
            print(f"Error al procesar exclusiones: {e}")

    # A. DENOMINADOR
    df_total_metas = df_validos.groupby(['ENTIDAD', 'VIGENCIA', 'Mes Programado'])['NOMBRE_META'].nunique().reset_index(name='Total_Metas_Diferentes')
    
    # B. NUMERADOR
    df_cumplen = df_validos[(df_validos['Cumple_Estructura'] == 'Sí') & (df_validos['Evidencia_Validada'] == 'Sí (Detectada)')]
    df_metas_cumplen = df_cumplen.groupby(['ENTIDAD', 'VIGENCIA', 'Mes Programado'])['NOMBRE_META'].nunique().reset_index(name='Metas_Cumplen')
    
    # C. COMBINAR
    df_resumen = pd.merge(df_total_metas, df_metas_cumplen, on=['ENTIDAD', 'VIGENCIA', 'Mes Programado'], how='left')
    df_resumen['Metas_Cumplen'] = df_resumen['Metas_Cumplen'].fillna(0) 
    df_resumen['% Avance Meta 8'] = ((df_resumen['Metas_Cumplen'] / df_resumen['Total_Metas_Diferentes']) * 100).round(2)
    df_resumen = df_resumen.sort_values(by=['VIGENCIA', 'Mes Programado', 'ENTIDAD'])
    
    # D. MATRIZ CRUZADA
    df_matriz = df_resumen.pivot_table(
        index='ENTIDAD',               
        columns=['VIGENCIA', 'Mes Programado'],   
        values='% Avance Meta 8'       
    )
    
    orden_localidades = [
        "01 - USAQUÉN", "02 - CHAPINERO", "03 - SANTA FE", "04 - SAN CRISTÓBAL", 
        "05 - USME", "06 - TUNJUELITO", "07 - BOSA", "08 - KENNEDY", 
        "09 - FONTIBÓN", "10 - ENGATIVÁ", "11 - SUBA", "12 - BARRIOS UNIDOS", 
        "13 - TEUSAQUILLO", "14 - LOS MÁRTIRES", "15 - ANTONIO NARIÑO", 
        "16 - PUENTE ARANDA", "17 - LA CANDELARIA", "18 - RAFAEL URIBE URIBE", 
        "19 - CIUDAD BOLÍVAR", "20 - SUMAPAZ"
    ]
    df_matriz = df_matriz.reindex(orden_localidades)

    # E. GUARDAR HOJAS
    with pd.ExcelWriter(nueva_ruta, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_resumen.to_excel(writer, sheet_name='Resumen Meta 8', index=False)
        df_matriz.to_excel(writer, sheet_name='Matriz Localidades', startcol=1)
        
        ws_matriz = writer.sheets['Matriz Localidades']
        ws_matriz.delete_rows(3)
        max_fila = ws_matriz.max_row
        ws_matriz.merge_cells(start_row=3, start_column=1, end_row=max_fila, end_column=1)
        
        celda_entidad = ws_matriz.cell(row=3, column=1, value="Entidad")
        celda_entidad.font = Font(bold=True, size=12)
        celda_entidad.alignment = Alignment(textRotation=90, horizontal='center', vertical='center')
        
        max_length = 0
        for cell in ws_matriz["B"]:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        ws_matriz.column_dimensions["B"].width = max(max_length + 3, 12)
        
    print(f"¡Proceso completado exitosamente!\nTu nuevo archivo es: {nueva_ruta}")

# Ejecutar
if __name__ == "__main__":
    ruta_datos = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\localidades_pto_20260929.xls"
    ruta_excl = r"C:\Users\Pc\OneDrive\Trabajo\2026\Secretaria de Gobierno\08 Septiembre\ObservacionesMeta8\Metas excluidas.xlsx" 
    
    procesar_sabana_observaciones(ruta_datos, ruta_exclusiones=ruta_excl)

Detectado archivo con extensión .xls. Verificando formato interno...
Es un archivo .xls antiguo. Convirtiendo datos (se perderá el color/formato original)...
Error: No se encontró la columna OBSERVACIONES.
